# 🎲 19 · Hidden Markov models I: decoding hidden states in sequences
### The forward, backward and Viterbi algorithms — derived line by line, checked against brute force, and used to find dice switches and CpG-island-like regions

**BiologyToML** · Level 1 · 02-620 core · Part D — Probabilistic graphical models · Module 19 of 44

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/xulabs/edu/blob/main/ml/tutorial/BiologyToML/notebooks/19_hmm_inference.ipynb)

| | |
|---|---|
| **Hardware target** | Google Colab free tier — CPU only (≈ 1 min to run everything); no GPU needed |
| **Core frameworks** | Python 3, NumPy, pandas, Matplotlib |
| **Prerequisites** | Modules 02 (probability: product rule, law of total probability, Bayes' rule), 15–16 (Bayesian networks, Markov blankets, conditional independence). Module 01 for NumPy; Module 03 for the vector–matrix product. |
| **Estimated time** | 185 min (reading + running + the ✏️ exercises) |
| **02-620 topics covered** | Hidden Markov models (part 1: model and inference): latent-variable models; speech, gene finding and haplotype phasing; the HMM as a Bayesian network (states, outputs, initial, transition and emission probabilities) and its joint probability; the two-dice example; Markov properties and Markov blankets; the inference tasks $P(Q\mid O)$, $P(q_t\mid O)$ and $\arg\max_Q P(Q\mid O)$; the forward and backward algorithms (with the worked example $O=\{2,3,6\}$); posterior decoding; the most probable path and the Viterbi algorithm |

> ⚠️ **Important notes.** For research and educational purposes only, not to be used for clinical diagnosis.
> The two-dice HMM is the worked example of the 02-620 lecture, typed in with its parameters (toolkit `DICE_HMM`). All sequences of dice rolls, the gene-structure illustration of Section 1 and the CpG-island-like genome of Section 11 are **SIMULATED**; no real genome is used. Every probability quoted in the text is printed by the code next to it.

## 🎯 Learning objectives

By the end of this notebook you will be able to:

1. **Explain** what a hidden Markov model (HMM) is — hidden states, outputs, initial, transition and emission probabilities — and draw it as a Bayesian network.
2. **Compute** the joint probability $P(Q, O)$ of a state path and an observation sequence, and **verify** the HMM's Markov properties numerically by brute-force enumeration.
3. **Derive** the forward and backward recursions line by line, naming the rule used at every step (law of total probability, product rule, Markov property), and **reproduce** the worked example $O=(2,3,6)$: $\alpha_1(A)=.14,\ \alpha_1(B)=.03,\ \alpha_2(A)=.0236,\ \alpha_2(B)=.0052$, and on to $P(O)=.004656$.
4. **Implement** the forward, backward, posterior-decoding and Viterbi algorithms from scratch, fill a Viterbi table with backpointers by hand, and **check** each algorithm against enumeration of all $n^T$ paths.
5. **Interpret** the difference between the probability of one path, the per-position posterior $P(q_t=s_i\mid O)$ and the single most probable path — and explain honestly why the Viterbi path of the classic example sequence `1 2 2 1 1 2 1 1 1 2 2` is *all A*.
6. **Explain** why naive HMM computations underflow on long sequences and **fix** it with scaling or log-space arithmetic.
7. **Apply** an HMM to segment a SIMULATED genome into CpG-island-like and background regions, and **compare** it with a sliding-window baseline.

## 🗺️ Contents

| Section | What you will do | ⏱ |
|---|---|---|
| 0 · Setup | imports, toolkit, the two-dice HMM | 5 min |
| 1 · Latent-variable models | observed vs hidden; a gene-structure picture; speech, haplotypes | 12 min |
| 2 · What is an HMM? | states, outputs, π, a, b; the HMM as a Bayesian network; parameter counting | 15 min |
| 3 · The two-dice example | simulate rolls; compute P(Q, O); verify the Markov properties | 20 min |
| 4 · Three questions and why brute force fails | the three inference tasks; nᵀ paths | 10 min |
| 5 · The forward algorithm | derivation, implementation, the O = {2, 3, 6} example, brute-force check | 25 min |
| 6 · The backward algorithm | derivation, implementation, Σ αβ = P(O) at every t | 15 min |
| 7 · P(Q | O) for one path | Bayes' rule; what is easy and what is hard | 10 min |
| 8 · Posterior decoding | P(qₜ = sᵢ | O) ∝ αβ; decoding a long simulated sequence | 15 min |
| 9 · The Viterbi algorithm | max instead of sum, backpointers; a table by hand; the example sequence, honestly | 25 min |
| 10 · Numerical underflow | scaling and log-space forward algorithms | 10 min |
| 11 · Biology: CpG-island-like regions | segmenting a SIMULATED genome; HMM vs sliding window | 15 min |
| Wrap-up | takeaways, checkpoint, questions, lab note | 10 min |

## 🧫 The biological question

In February 2001 the first draft of the human genome was published: about three billion letters
`…cggtgaaactgcacgattgttgctggcttaaagatagacc…` (Lander *et al.* 2001). The letters were known; their *meaning* was not.
**Where are the genes?** Which stretches are **exons** (kept in the messenger RNA), which are **introns** (spliced out),
and which are **intergenic** background? Nobody had labelled the genome — the labels are **hidden** — yet they leave
statistical fingerprints in the sequence: exons have a codon structure, intron boundaries carry splice signals,
and some regions are unusually rich in C and G. And the labels have a *grammar*: an exon is followed by an intron or by
background, not by a random jumble, and each label tends to persist over many consecutive bases.

A **hidden Markov model** captures exactly these two facts: (1) hidden labels follow each other with certain
probabilities (a Markov chain), and (2) each label "speaks its own dialect" of A, C, G, T. Given a sequence, the
algorithms of this notebook compute *how likely each label is at each position* and *the single best labelling*.
The same machinery decodes words from sound in speech recognition, finds protein families (profile HMMs, Module 20),
phases haplotypes from genotypes, segments tumour genomes into copy-number states, and annotates chromatin states from
histone marks (ChromHMM; Ernst & Kellis 2012). The 02-620 lecture teaches it with a toy that is easier to think about —
**a croupier who secretly switches between two dice** — and so will we, before returning to DNA in Section 11.

> 🧪 **Simulated data, with a known truth.** The course stores no data sets: every sequence in this notebook (dice rolls,
> the gene-structure picture, the CpG-island-like genome) is generated by code from an HMM whose parameters we know.
> That is not only convenient — it means we can always compare a decoding with the *true* hidden states, a check that
> is impossible on a real genome, where nobody knows the labels.

---
## 0 · Environment setup

Run the cells **in order, from the top** (Shift + Enter). The first cell reports your hardware; this module needs only a CPU.

In [ ]:
# ── Hardware check ──
import platform
import subprocess
import sys

print("Python", sys.version.split()[0], "on", platform.system())
try:
    gpu = subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
                         capture_output=True, text=True, timeout=10).stdout.strip()
    print("GPU:", gpu or "none detected — that is fine, a CPU is enough for this course.")
except (FileNotFoundError, subprocess.TimeoutExpired):
    print("GPU: none detected — that is fine, a CPU is enough for this course.")

In [ ]:
# ── Dependencies (Colab already has these; we install only what is missing) ──
import importlib

for module, pip_name in [("numpy", "numpy"), ("pandas", "pandas"), ("matplotlib", "matplotlib")]:
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pip_name], check=True)
print("✅ All dependencies are available.")

In [ ]:
# ── Global imports and configuration ──
import itertools
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Circle, FancyArrowPatch

SEED = 0                                   # fixing the random seed makes every run identical
rng = np.random.default_rng(SEED)
plt.rcParams.update({"figure.dpi": 90, "figure.figsize": (6.4, 3.6), "axes.titlesize": 10,
                     "axes.labelsize": 9, "legend.fontsize": 8, "axes.spines.top": False,
                     "axes.spines.right": False})
print("NumPy", np.__version__, "| seed", SEED)

### 🧰 Course toolkit

The next cell copies the shared helpers this notebook needs from [`course_toolkit.py`](../course_toolkit.py).
`DICE_HMM` holds the two-dice model of the 02-620 lecture (initial, transition and emission probabilities); `logsumexp` adds probabilities stored as logarithms without underflow (Section 10); `accuracy` and `precision_recall_f1` score a decoding against the truth; `draw_graph` draws small graphs. Just run it.

In [ ]:
# ── Course toolkit (identical to ../course_toolkit.py v2.0; copied so this notebook runs alone) ──
import os
import numpy as np
import matplotlib.pyplot as plt

DICE_HMM = {
    "states": ("A", "B"),
    "faces": (1, 2, 3, 4, 5, 6),
    "pi": np.array([0.7, 0.3]),                              # initial state probabilities
    "A": np.array([[0.8, 0.2],                               # A[i, j] = P(q_t = j | q_{t-1} = i)
                   [0.2, 0.8]]),
    "B": np.array([[0.3, 0.2, 0.2, 0.1, 0.1, 0.1],           # B[i, k] = P(o_t = face k+1 | q_t = i)
                   [0.1, 0.1, 0.1, 0.2, 0.2, 0.3]]),
}


def logsumexp(a, axis=None):
    """log(Σ exp(a)) computed stably (subtract the maximum first)."""
    a = np.asarray(a, float)
    m = np.max(a, axis=axis, keepdims=True)
    m = np.where(np.isfinite(m), m, 0.0)
    out = np.log(np.sum(np.exp(a - m), axis=axis, keepdims=True)) + m
    return np.squeeze(out, axis=axis) if axis is not None else out.item()


def accuracy(y_true, y_pred) -> float:
    """Fraction of correct predictions."""
    return float(np.mean(np.asarray(y_true) == np.asarray(y_pred)))


def confusion_counts(y_true, y_pred) -> dict:
    """True/false positives/negatives for binary labels (1 = positive)."""
    y_true, y_pred = np.asarray(y_true).astype(int), np.asarray(y_pred).astype(int)
    return dict(TP=int(np.sum((y_pred == 1) & (y_true == 1))), FP=int(np.sum((y_pred == 1) & (y_true == 0))),
                FN=int(np.sum((y_pred == 0) & (y_true == 1))), TN=int(np.sum((y_pred == 0) & (y_true == 0))))


def precision_recall_f1(y_true, y_pred):
    """Precision TP/(TP+FP), recall TP/(TP+FN) and their harmonic mean F1 (0 when undefined)."""
    c = confusion_counts(y_true, y_pred)
    precision = c["TP"] / (c["TP"] + c["FP"]) if c["TP"] + c["FP"] else 0.0
    recall = c["TP"] / (c["TP"] + c["FN"]) if c["TP"] + c["FN"] else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0
    return precision, recall, f1


def draw_graph(edges, pos: dict, directed: bool = True, ax=None, highlight=None, node_color: str = "#9ecae1",
               highlight_color: str = "#fdae6b", radius: float = 0.07, title: str = "", edge_labels=None):
    """Draw a small graph with Matplotlib only.

    ``edges`` is a list of (u, v) pairs, ``pos`` maps every node to (x, y) in [0, 1]².
    ``highlight`` is a set of nodes to colour differently (e.g. observed evidence).
    """
    from matplotlib.patches import Circle, FancyArrowPatch
    ax = ax or plt.gca()
    highlight = set(highlight or [])
    for i, (u, v) in enumerate(edges):
        (x0, y0), (x1, y1) = pos[u], pos[v]
        d = np.hypot(x1 - x0, y1 - y0) or 1.0
        sx, sy = x0 + (x1 - x0) * radius / d, y0 + (y1 - y0) * radius / d
        ex, ey = x1 - (x1 - x0) * radius / d, y1 - (y1 - y0) * radius / d
        ax.add_patch(FancyArrowPatch((sx, sy), (ex, ey), arrowstyle="-|>" if directed else "-",
                                     mutation_scale=12, lw=1.2, color="0.25"))
        if edge_labels is not None:
            ax.text((x0 + x1) / 2, (y0 + y1) / 2, str(edge_labels[i]), fontsize=7, color="0.2",
                    ha="center", va="center", bbox=dict(fc="white", ec="none", alpha=0.7, pad=0.5))
    for node, (x, y) in pos.items():
        ax.add_patch(Circle((x, y), radius, fc=highlight_color if node in highlight else node_color,
                            ec="0.2", lw=1, zorder=3))
        ax.text(x, y, str(node), ha="center", va="center", fontsize=9, zorder=4)
    ax.set_xlim(-0.1, 1.1); ax.set_ylim(-0.1, 1.1); ax.set_aspect("equal"); ax.axis("off")
    if title:
        ax.set_title(title)
    return ax


print("🧰 Toolkit loaded:", ", ".join(['logsumexp', 'accuracy', 'confusion_counts', 'precision_recall_f1', 'draw_graph']))

---
## 1 · Latent-variable models: when the most interesting variable is not measured

> 📍 **Where we are.** In [Modules 15–17](15_bayesian_networks_representation.ipynb) every variable of a Bayesian network
> could, in principle, be observed. In [Module 13](13_mixture_models_em.ipynb) (mixture models) we met a **latent** (hidden) variable: the cluster label of each sample. An HMM is the
> sequential version of that idea: one hidden label *per position*, and the labels of neighbouring positions are linked.

Picture a data table (left panel of the next figure). Each row is a sample, each column a variable — but some columns
are **never observed**. In a mixture model the hidden column is the cluster of each cell or patient. In sequence
problems the hidden column is a label at every position of the sequence:

| Application | Observations $o_t$ | Hidden states $q_t$ |
|---|---|---|
| Speech processing | sound signal in each 10-ms frame | parts of speech, words |
| Gene finding (one of the earliest and most influential uses of HMMs in biology; HMM gene finders such as GENSCAN and Genie helped annotate the 2001 draft human genome) | nucleotides A, C, G, T | exon / intron / intergenic |
| Haplotype phasing (a haplotype is the collection of alleles carried on the *same* chromosome) | genotypes at consecutive markers (unordered allele pairs) | which parental chromosome ("phase") each allele came from |
| This notebook's toy | the face of a die, 1–6 | which die (A or B) was used |

In every row, an HMM models the observations with a set of hidden states that do two things: **hidden states
generate observations**, and **hidden states transition to other hidden states**. The plan of this lecture topic is:
the model; inference (the forward–backward and Viterbi algorithms — this notebook); learning (the EM or Baum–Welch
algorithm) and dynamic Bayesian networks, which generalise HMMs ([Module 20](20_hmm_learning.ipynb)).

The right panel of the next figure shows what gene finding is about, on a SIMULATED stretch of DNA: the letters are
all we observe; the gene structure (which letters belong to exons, introns and the intergenic background) is hidden.

In [ ]:
# ── 1.0 Picture: hidden columns, and genes hidden in a genome (SIMULATED) ──
rng = np.random.default_rng(SEED + 1)
segments = [("B", 16), ("E", 9), ("I", 13), ("E", 8), ("B", 14), ("E", 10), ("I", 9), ("E", 7), ("I", 8), ("E", 9), ("B", 9)]
labels = "".join(lab * n for lab, n in segments)               # hidden: B = background, E = exon, I = intron
compo = {"B": [0.30, 0.20, 0.20, 0.30], "E": [0.20, 0.30, 0.30, 0.20], "I": [0.32, 0.18, 0.18, 0.32]}
dna = "".join(rng.choice(list("acgt"), p=compo[lab]) for lab in labels)   # observed letters
LAB_COLORS = {"B": "tab:blue", "E": "tab:red", "I": "goldenrod"}

fig = plt.figure(figsize=(12, 3.4))
ax0 = fig.add_axes([0.01, 0.12, 0.17, 0.72])                  # the latent-variable data table
ax0.add_patch(plt.Rectangle((0, 0), 0.62, 1, fc="#fdb863", ec="0.3"))
ax0.add_patch(plt.Rectangle((0.62, 0), 0.38, 1, fc="#80b1d3", ec="0.3"))
ax0.text(0.31, 0.5, "observed", ha="center", va="center", fontsize=9)
ax0.text(0.81, 0.5, "unobserved", ha="center", va="center", fontsize=8, rotation=90)
ax0.text(0.5, 1.05, "variables / features", ha="center", fontsize=8)
ax0.text(-0.06, 0.5, "samples", rotation=90, ha="center", va="center", fontsize=8)
ax0.set_xlim(-0.12, 1.02); ax0.set_ylim(-0.02, 1.15); ax0.axis("off")
ax0.set_title("a latent-variable model", fontsize=9)

ax1 = fig.add_axes([0.22, 0.05, 0.77, 0.85])
L = len(labels)
ax1.plot([0, L], [3.0, 3.0], color="tab:blue", lw=2, zorder=0)  # intergenic: a thin line (behind the boxes)
start = 0
for lab, n in segments:
    if lab != "B":
        h = 0.5 if lab == "E" else 0.25                           # exons tall, introns thin
        ax1.add_patch(plt.Rectangle((start, 3.0 - h / 2), n, h, fc=LAB_COLORS[lab], ec="0.3", lw=0.8))
    start += n
for g, (a, b) in enumerate([(16, 46), (60, 103)], 1):         # gene extents (first exon start → last exon end)
    ax1.annotate("", xy=(a, 3.55), xytext=(b, 3.55), arrowprops=dict(arrowstyle="|-|", lw=0.8))
    ax1.text((a + b) / 2, 3.7, f"gene {g}", ha="center", fontsize=9)
per_row = 63
for k, (ch, lab) in enumerate(zip(dna, labels)):               # the observed sequence, coloured by its hidden label
    ax1.text((k % per_row) * L / per_row, 1.9 - 0.75 * (k // per_row), ch, color=LAB_COLORS[lab],
             fontsize=10, family="monospace", ha="left", va="center")
for lab, name in [("B", "intergenic (B)"), ("I", "intron (I)"), ("E", "exon (E)")]:
    ax1.plot([], [], "s", color=LAB_COLORS[lab], label=name)
ax1.legend(loc="lower right", ncol=3, fontsize=8, frameon=False, bbox_to_anchor=(1.0, -0.12))
ax1.set_xlim(-1, L + 1); ax1.set_ylim(0.6, 4.0); ax1.axis("off")
ax1.set_title("SIMULATED DNA: where are the genes? (in real data the colours — the hidden states — are not shown)",
              fontsize=9)
plt.show()
print("hidden labels:", labels[:63], "…")
print("observed DNA :", dna[:63], "…")

🔍 **What just happened?** The left panel is the abstract picture of every latent-variable model: a data table with
some columns we can never measure. The right panel is the gene-finding version: two simulated genes, each made of exons
(red, kept in the messenger RNA) and introns (gold, spliced out), separated by intergenic background (blue). A
sequencing machine gives us only the letters; an annotation wants the colours. The colours leave weak statistical
fingerprints (here exons are slightly richer in C and G) and follow a grammar (exon, intron, exon, …, background) —
exactly the two ingredients an HMM is built from.

A short gene-finding example writes one hidden label above each nucleotide (**B** = background, **E** = exon,
**I** = intron):

```text
States:        B E E E I I I E E E B B
Observations:  A T T C G A T A T C G G
```

Two features of such labels are worth noticing *before* any mathematics. Let us look at them in code.

In [ ]:
# ── 1.1 Hidden labels come in runs ──
states_s9 = "BEEEIIIEEEBB"          # hidden labels (B background, E exon, I intron)
obs_s9 = "ATTCGATATCGG"             # the observed nucleotides

def run_lengths(labels: str) -> list[tuple[str, int]]:
    """Split a label string into runs of identical labels: 'BEEE' → [('B', 1), ('E', 3)]."""
    runs = []
    for lab, group in itertools.groupby(labels):
        runs.append((lab, len(list(group))))
    return runs

print("states      ", " ".join(states_s9))
print("observations", " ".join(obs_s9))
print("runs of identical hidden labels:", run_lengths(states_s9))
n_switch = sum(a != b for a, b in zip(states_s9[:-1], states_s9[1:]))
print(f"{n_switch} label changes in {len(states_s9) - 1} neighbouring pairs "
      f"→ a label is followed by the same label {1 - n_switch / (len(states_s9) - 1):.0%} of the time")

🔍 **What just happened?** Even in this 12-letter cartoon, the hidden labels form **runs**: 7 of the 11 neighbouring
pairs keep the same label. In a real genome an exon is ~100–200 bases long and an intron often thousands, so the
persistence is much stronger. This is the first ingredient of an HMM: **the label at position $t+1$ depends on the label
at position $t$** (and, as an approximation, on nothing earlier). The second ingredient is that **each label produces
letters with its own probabilities** — e.g. exons and CpG islands are richer in G and C than the genome average.
An HMM is nothing more than these two ingredients written as probability tables.

✏️ **Your turn.** Write your own 20-letter label string for a gene with two exons and one intron (use B, E, I) and run
`run_lengths` on it. How many label changes does a gene with $k$ exons need, at minimum? (Answer: background → exon,
then exon ↔ intron alternations, then exon → background: $2k$ changes for $k$ exons and $k-1$ introns.)

### 📘 Lecture notes — latent-variable models and HMMs

**Latent-variable models.** Some variables (columns of the data table) are observed, others are never observed. A
mixture model (Module 13) has one hidden cluster label per sample; an HMM has one hidden state per position of a
sequence, and neighbouring states are linked.

**Examples.** *Speech processing:* observations = sound signals, hidden states = parts of speech, words.
*Gene finding* — one of the earliest and most influential uses of HMMs in biology (HMM gene finders such as GENSCAN
and Genie helped annotate the first draft of the human genome, 2001): observations =
nucleotides, hidden states = introns / exons / intergenic regions ("where are the genes?"). *Haplotype phasing:*
haplotypes are collections of alleles derived from the same chromosome; from unphased genotypes, infer which allele
sits on which chromosome (the hidden phase).

**An HMM** models a set of observations with a set of hidden states: (1) hidden states **generate** observations
(emission); (2) hidden states **transition** to other hidden states. Gene-finding example:

```text
States:        B E E E I I I E E E B B
Observations:  A T T C G A T A T C G G
```

**Plan of the topic.** Model; inference — forward–backward and Viterbi algorithms (this module); learning — the EM
(Baum–Welch) algorithm; dynamic Bayesian networks, a generalisation of HMMs (Module 20).

> 🎓 **Exam tip.** Be able to say, for any application, what the observations and the hidden states are — exam questions often start with "define $S$ and $\Sigma$ for this problem". An HMM is "one mixture model per position, with the cluster labels chained together".

---
## 2 · What is a hidden Markov model?

> 📍 **Where we are.** We have the intuition (runs of hidden labels, each emitting letters in its own way). Now we write
> it as a probability model — a Bayesian network of a very particular shape ([Module 15](15_bayesian_networks_representation.ipynb)).

### 2.1 Random variables

* $q_1, q_2, \dots, q_T$ are the **states** (hidden). Each takes a value in a finite set $S=\{s_1,\dots,s_n\}$ of
  $n$ possible states, e.g. $S=\{I, E, B\}$ for gene finding or $S=\{A, B\}$ for the two dice.
* $o_1, o_2, \dots, o_T$ are the **outputs** or **observations**. Each takes a value in a finite **alphabet**
  $\Sigma=\{\sigma_1,\dots,\sigma_m\}$ of $m$ symbols, e.g. $\{A, C, G, T\}$ ($m=4$) or the die faces $\{1,\dots,6\}$ ($m=6$).
* $T$ is the length of the sequence; the index $t$ is "time" or "position along the DNA".

### 2.2 The three conditional probability tables

We have defined the random variables; how do we define the conditional probability distribution of each? Ask how
many parents each variable has in the Bayesian network: $q_1$ has **no parent**, each $q_t$ ($t \ge 2$) has **one
parent** $q_{t-1}$, and each $o_t$ has **one parent** $q_t$. So we need exactly three kinds of tables:

| Name | Definition | Size | Dice example |
|---|---|---|---|
| **initial state probability** | $\pi_i = P(q_1 = s_i)$ | $n$ numbers | $\pi_A = 0.7,\ \pi_B = 0.3$ |
| **transition probability** | $a_{ij} = P(q_t = s_j \mid q_{t-1} = s_i)$ | $n\times n$ matrix | stay 0.8, switch 0.2 |
| **emission probability** | $b_i(\sigma) = P(o_t = \sigma \mid q_t = s_i)$ | $n\times m$ matrix | die A favours 1, die B favours 6 |

The same transition matrix is used at every step and the same emission table at every position:
$P(q_2\mid q_1)=P(q_3\mid q_2)=\dots=P(q_T\mid q_{T-1})$ and $P(o_1\mid q_1)=P(o_2\mid q_2)=\dots=P(o_T\mid q_T)$. This
**sharing** is what keeps the model small.

⚠️ **Index order.** Row $i$ = where you come *from*, column $j$ = where you go *to*: $a_{ij}=P(\text{to } s_j\mid \text{from } s_i)$,
so every **row** sums to 1. This is the convention of the toolkit and of the counting formulas of Module 20. You will
also meet the same probability written as $P(q_t=s_i\mid q_{t-1}=s_j)$ — with the letters $i$ and $j$ swapping roles.
Always ask "which index is the previous state?" before using a formula.

🧮 **Math refresher — Σ and ∏.** $\sum_{t=1}^{T} x_t = x_1 + x_2 + \dots + x_T$ ("add up"), and
$\prod_{t=2}^{T} x_t = x_2 \times x_3 \times \dots \times x_T$ ("multiply together"). A conditional probability
$P(X \mid Y)$ is the probability of $X$ in the world where $Y$ is known to have happened; the **product rule** (Module 02)
says $P(X, Y) = P(X\mid Y)\,P(Y)$.

### 2.3 The HMM as a Bayesian network

Let us draw the graph.

In [ ]:
# ── 2.1 Draw the HMM as a Bayesian network ──
T_draw = 4
pos = {}
for t in range(1, T_draw + 1):
    x = 0.1 + 0.8 * (t - 1) / (T_draw - 1)
    pos[f"q{t}"] = (x, 0.75)                  # hidden states on top
    pos[f"o{t}"] = (x, 0.25)                  # observations below
edges = [(f"q{t}", f"q{t + 1}") for t in range(1, T_draw)] + [(f"q{t}", f"o{t}") for t in range(1, T_draw + 1)]

fig, ax = plt.subplots(figsize=(6, 2.6))
draw_graph(edges, pos, ax=ax, highlight={f"o{t}" for t in range(1, T_draw + 1)}, radius=0.06,
           title="HMM as a Bayesian network (orange = observed outputs, blue = hidden states)")
ax.set_ylim(0.1, 0.9)
plt.tight_layout(); plt.show()

parents = {v: [u for u, w in edges if w == v] for v in pos}
for v in ["q1", "q2", "q3", "o1", "o3"]:
    print(f"parents of {v}: {parents[v] or 'none'}")

🔍 **What just happened?** The graph is a chain of hidden states with one observed "leaf" hanging below each. The
Bayesian-network factorisation rule of Module 15 — *the joint distribution is the product of each variable's
probability given its parents* — therefore factorises the joint distribution as:

$$P(Q, O) \;=\; \underbrace{P(q_1)}_{\text{initial}}\;\underbrace{P(o_1 \mid q_1)}_{\text{emission}}\;
\prod_{t=2}^{T} \underbrace{P(q_t \mid q_{t-1})}_{\text{transition}}\;\underbrace{P(o_t \mid q_t)}_{\text{emission}},
\qquad Q = (q_1,\dots,q_T),\; O = (o_1,\dots,o_T).$$

*In words:* the probability that the hidden labels were $Q$ **and** the observations were $O$ is: the probability of
the first label, times the probability that it emitted the first observation, times — for every later position — the
probability of moving to the next label and of that label emitting the next observation. In the notation of
Section 2.2: $P(Q,O)=\pi_{q_1}\,b_{q_1}(o_1)\prod_{t=2}^T a_{q_{t-1}q_t}\,b_{q_t}(o_t)$.

### 2.4 Conditional independence buys tractability

How many numbers would we need to describe the joint distribution of $T$ (state, output) pairs *without* any
independence assumption? Each position has $n\cdot m$ possible (state, output) combinations, so the joint table has
$(nm)^T$ cells, minus one because they must sum to 1. The HMM needs only $(n-1) + n(n-1) + n(m-1)$ free numbers —
independent of $T$.

In [ ]:
# ── 2.2 Free parameters: full joint table vs HMM ──
def hmm_free_parameters(n: int, m: int) -> int:
    """Number of free numbers in an HMM: (n−1) initial + n(n−1) transition + n(m−1) emission."""
    return (n - 1) + n * (n - 1) + n * (m - 1)

n_dice, m_dice = 2, 6
print(f"HMM with n={n_dice} states, m={m_dice} faces: {hmm_free_parameters(n_dice, m_dice)} free parameters, for any T")
for T in [1, 2, 3, 5, 10, 20]:
    full = (n_dice * m_dice) ** T - 1
    print(f"T = {T:2d}: full joint table needs {full:.3g} free parameters")
print(f"gene finder (n=3 labels, m=4 bases): {hmm_free_parameters(3, 4)} free parameters")

🔍 **What just happened?** For the dice, the HMM uses **13** numbers whatever the length of the sequence, while a
full joint table over just 10 rolls would need about $6\times 10^{10}$. This is the course's recurring theme —
*conditional independence buys tractability* — in its sequential form: 3ⁿ → 4n + 1 for naïve Bayes (Module 08), 31 → 10 for
the alarm network (Module 15), and here "exponential in $T$" → "constant in $T$". The price is an assumption: the
future depends on the past **only through the current state**. Section 3.3 checks what that means numerically.

✏️ **Your turn.** A gene finder that reads DNA one **codon** at a time has $m = 64$ output symbols. How many free
parameters does a 3-state HMM need then? And a 6-state model (exon in each of three reading frames, intron, background,
start region)? Use `hmm_free_parameters`. Which part of the count grows fastest with $n$?

---
## 3 · The two-dice example

> 📍 **Where we are.** We have the general definition. Now we load the running example of this lecture topic, generate
> data from it, compute $P(Q, O)$ for a given path, and check the Markov properties by brute force.

A croupier rolls one of two six-sided dice at each turn. Die **A** is biased towards low faces, die **B** towards high
faces. The current die depends on the previous one: after each roll the croupier keeps the same die with probability
0.8 or switches with probability 0.2. The first die is A with probability $\pi_A=0.7$ and B with $\pi_B=0.3$. The two
emission tables are

| face $v$ | 1 | 2 | 3 | 4 | 5 | 6 |
|---|---|---|---|---|---|---|
| $P(v\mid A)$ | .3 | .2 | .2 | .1 | .1 | .1 |
| $P(v\mid B)$ | .1 | .1 | .1 | .2 | .2 | .3 |

We see the faces; we do not see which die was used. (This is the 02-620 version of the "occasionally dishonest casino"
of Durbin *et al.* 1998, Chapter 3.)

⚠️ **A naming clash to avoid.** The dice are called A and B, and many textbooks (Rabiner 1989) *also* call the transition
matrix $A$ and the emission matrix $B$ — the toolkit's `DICE_HMM["A"]` and `DICE_HMM["B"]` follow that tradition. To keep
our heads clear we copy them into variables called `trans` and `emit` and reserve "A" and "B" for the dice.

In code, faces are stored as **indices 0–5** (face 1 → index 0, …, face 6 → index 5) because NumPy arrays are indexed
from 0. Likewise, Python position `t = 0` is the time $t=1$ of the formulas. We will convert back whenever we print.
The next cell draws the **state-transition diagram**, the second way of picturing an HMM: nodes are *values* of a state
(not random variables), arrows are transition probabilities.

In [ ]:
# ── 3.1 The two-dice HMM ──
STATES = DICE_HMM["states"]            # ('A', 'B')
FACES = DICE_HMM["faces"]              # (1, 2, 3, 4, 5, 6)
pi = DICE_HMM["pi"]                    # π_i      = P(q_1 = s_i)
trans = DICE_HMM["A"]                  # a_ij     = P(q_t = s_j | q_{t-1} = s_i)   (row = from, column = to)
emit = DICE_HMM["B"]                   # b_i(k)   = P(o_t = face k | q_t = s_i)

print("initial probabilities π:", {s: float(p) for s, p in zip(STATES, pi)})
print("\ntransition matrix (row = die at t, column = die at t+1):")
print(pd.DataFrame(trans, index=STATES, columns=STATES))
print("\nemission probabilities (one row per die):")
print(pd.DataFrame(emit, index=[f"P(v|{s})" for s in STATES], columns=FACES))
print("\nrows sum to 1?", np.allclose(trans.sum(1), 1), np.allclose(emit.sum(1), 1), np.isclose(pi.sum(), 1))
print(f"mean face: die A = {emit[0] @ np.array(FACES):.1f} | die B = {emit[1] @ np.array(FACES):.1f}")

In [ ]:
# ── 3.2 State-transition diagram ──
DIE_COLORS = ("tab:blue", "tab:orange")

def draw_state_diagram(ax, pi, trans, names=("A", "B"), colors=DIE_COLORS):
    """Two-state transition diagram: circles for states, curved arrows for switches, loops for staying."""
    xy = [(0.3, 0.5), (0.7, 0.5)]
    r = 0.09
    for i, (x, y) in enumerate(xy):
        ax.add_patch(Circle((x, y), r, fc=colors[i], alpha=0.35, ec="0.2", lw=1.2))
        ax.text(x, y, names[i], ha="center", va="center", fontsize=13, weight="bold")
        ax.add_patch(FancyArrowPatch((x - 0.045, y + 0.075), (x + 0.045, y + 0.075), connectionstyle="arc3,rad=-2.4",
                                     arrowstyle="-|>", mutation_scale=12, lw=1.2, color="0.25"))
        ax.text(x, y + 0.27, f"stay {trans[i, i]:.1f}", ha="center", fontsize=9)
        ax.add_patch(FancyArrowPatch((x, y - 0.3), (x, y - r - 0.01), arrowstyle="-|>", mutation_scale=10,
                                     lw=1, color="0.5", linestyle="--"))
        ax.text(x, y - 0.32, f"start π = {pi[i]:.1f}", ha="center", va="top", fontsize=8, color="0.3")
    (x0, y0), (x1, y1) = xy
    ax.add_patch(FancyArrowPatch((x0 + r * 0.8, y0 + 0.05), (x1 - r * 0.8, y1 + 0.05), connectionstyle="arc3,rad=-0.35",
                                 arrowstyle="-|>", mutation_scale=12, lw=1.2, color="0.25"))
    ax.text(0.5, y0 + 0.14, f"{trans[0, 1]:.1f}", ha="center", fontsize=9)
    ax.add_patch(FancyArrowPatch((x1 - r * 0.8, y1 - 0.05), (x0 + r * 0.8, y0 - 0.05), connectionstyle="arc3,rad=-0.35",
                                 arrowstyle="-|>", mutation_scale=12, lw=1.2, color="0.25"))
    ax.text(0.5, y0 - 0.17, f"{trans[1, 0]:.1f}", ha="center", fontsize=9)
    ax.set_xlim(0.05, 0.95); ax.set_ylim(0.05, 0.9); ax.set_aspect("equal"); ax.axis("off")


fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(9, 3), gridspec_kw={"width_ratios": [1, 1.3]})
draw_state_diagram(ax1, pi, trans)
ax1.set_title("state-transition diagram")
for i, c in enumerate(DIE_COLORS):
    ax2.bar(np.array(FACES) + (i - 0.5) * 0.38, emit[i], width=0.38, color=c, label=f"die {STATES[i]}")
ax2.set_xlabel("face"); ax2.set_ylabel("emission probability"); ax2.set_xticks(FACES)
ax2.set_title("emission probabilities: A favours 1, B favours 6"); ax2.legend()
plt.tight_layout(); plt.show()

### 3.1 The generative story: simulating rolls

An HMM is a **generative model**: it tells a story of how data come about. To *sample* a sequence: draw $q_1$ from $\pi$;
then repeatedly draw the next die from the row of `trans` belonging to the current die; finally, at each position, draw
a face from the row of `emit` belonging to that position's die. To draw from a discrete distribution we use the
**inverse-CDF trick**: draw a uniform number $u \in [0,1)$ and pick the first category whose cumulative probability
exceeds $u$ — the many-category version of the `rng.random() < p` coin flip used since Module 02.

In [ ]:
# ── 3.3 Simulate dice rolls from the HMM ──
def sample_hmm(T: int, pi, trans, emit, rng):
    """Sample hidden states and observations (both as 0-based integer arrays of length T) from an HMM."""
    n = len(pi)
    cum_trans = np.cumsum(trans, axis=1)
    u = rng.random(T)
    states = np.empty(T, dtype=int)
    states[0] = rng.choice(n, p=pi)
    for t in range(1, T):                                   # the Markov chain must be generated in order
        states[t] = min(np.searchsorted(cum_trans[states[t - 1]], u[t], side="right"), n - 1)
    cum_emit = np.cumsum(emit[states], axis=1)              # each position's emission CDF
    obs = np.minimum((rng.random(T)[:, None] >= cum_emit).sum(axis=1), emit.shape[1] - 1)
    return states, obs


rng = np.random.default_rng(SEED + 3)
q_sim, o_sim = sample_hmm(80, pi, trans, emit, rng)
print("dice :", "".join(STATES[s] for s in q_sim))
print("faces:", "".join(str(f + 1) for f in o_sim))
print(f"fraction of rolls with die B: {np.mean(q_sim == 1):.2f};  mean face with A: {o_sim[q_sim == 0].mean() + 1:.2f}, "
      f"with B: {o_sim[q_sim == 1].mean() + 1:.2f}")

fig, ax = plt.subplots(figsize=(10, 2.4))
ax.bar(np.arange(1, 81), o_sim + 1, color=[DIE_COLORS[s] for s in q_sim], width=0.85)
ax.set_xlabel("roll number t"); ax.set_ylabel("face $o_t$"); ax.set_yticks(FACES)
ax.set_title("SIMULATED: 80 rolls coloured by the hidden die (blue = A, orange = B) — in real data the colours are invisible")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The dice come in runs (because staying has probability 0.8, a run lasts on average
$1/(1-0.8)=5$ rolls — Module 20 derives this), and orange runs contain more high faces. But look at the bars without
the colours: individual rolls are ambiguous — die A also rolls 6s (probability 0.1) and die B rolls 1s (0.1). Recovering
the colours requires combining *many* neighbouring rolls with the knowledge that dice persist. That is what inference
in an HMM does.

✏️ **Your turn.** Re-run the cell with a stickier croupier, `trans_sticky = np.array([[0.95, 0.05], [0.05, 0.95]])`
passed to `sample_hmm`. How do the runs change? Which do you expect to be easier to decode, and why?

### 3.2 The probability of one path and one sequence

Here is one run of the croupier's game — the hidden dice and the rolls they produced (we call it the *story
example*):

```text
Hidden states:  A A A B B B B B B A A
Observations:   1 6 2 1 4 2 1 1 1 2 5
```

Let us compute $P(Q, O)$ with the formula of Section 2.3, printing every factor.

In [ ]:
# ── 3.4 Joint probability P(Q, O) of a path and a sequence ──
def to_states(s: str) -> np.ndarray:
    """'AAB' → array([0, 0, 1])."""
    return np.array([STATES.index(ch) for ch in s.replace(" ", "")])

def to_obs(s: str) -> np.ndarray:
    """'1 6 2' → array([0, 5, 1]) (faces → 0-based indices)."""
    return np.array([int(ch) - 1 for ch in s.split()])

def joint_prob(states, obs, pi, trans, emit, verbose=False) -> float:
    """P(Q, O) = π_{q1} b_{q1}(o1) ∏_{t≥2} a_{q_{t-1} q_t} b_{q_t}(o_t)."""
    p = pi[states[0]] * emit[states[0], obs[0]]
    if verbose:
        print(f"t=1: π_{STATES[states[0]]} · b_{STATES[states[0]]}({obs[0] + 1}) = {pi[states[0]]} · {emit[states[0], obs[0]]}")
    for t in range(1, len(obs)):
        a, b = trans[states[t - 1], states[t]], emit[states[t], obs[t]]
        if verbose:
            print(f"t={t + 1}: a_{STATES[states[t - 1]]}{STATES[states[t]]} · b_{STATES[states[t]]}({obs[t] + 1}) = {a} · {b}")
        p *= a * b
    return p


Q21, O21 = to_states("AAABBBBBBAA"), to_obs("1 6 2 1 4 2 1 1 1 2 5")
p_story = joint_prob(Q21, O21, pi, trans, emit, verbose=True)
p_allA = joint_prob(to_states("A" * 11), O21, pi, trans, emit)
print(f"\nP(Q = AAABBBBBBAA, O) = {p_story:.4g}")
print(f"P(Q = AAAAAAAAAAA, O) = {p_allA:.4g}   ({p_allA / p_story:.0f} times larger)")

🔍 **What just happened?** The joint probability of the story example is about $1.1\times10^{-12}$ — tiny, because it
multiplies 22 numbers below 1 (one transition and one emission per roll, plus the start). Tiny joint probabilities are
normal; what matters is how they *compare*. Surprise: the path "all A" explains the *same* rolls about **1,300 times
better** than the path that, in the story, actually produced them. The faces 1, 2, 1, 1, 1 in the B-run are
much more typical of die A. Keep this in mind: **the true hidden path is not necessarily the most probable one** — the
data may simply not contain enough evidence for it. We return to this in Section 9.

Before going on, a sanity check that the formula defines a proper probability distribution: summed over every possible
path *and* every possible observation sequence, $P(Q,O)$ must give exactly 1. For $T = 3$ that is $2^3 \times 6^3 = 1{,}728$ terms.

In [ ]:
# ── 3.5 Sanity check: P(Q, O) sums to 1 over all paths and all sequences (T = 3) ──
total = 0.0
for Q in itertools.product(range(2), repeat=3):
    for O in itertools.product(range(6), repeat=3):
        total += joint_prob(Q, O, pi, trans, emit)
print(f"sum over 2^3 paths × 6^3 sequences = {total:.12f}")

### 3.3 Markov properties, checked by brute force

The **Markov property**: *given the current state, the next state is independent of all earlier states*,
$q_{t+1} \perp (q_{t-1},\dots,q_1) \mid q_t$. More generally, the independence statements of an HMM can be read off the
graph with the tools of [Module 16](16_bayesian_networks_independence.ipynb) — e.g. the **Markov blanket** of a node (parents, children and the children's other
parents) makes it independent of everything else. For $q_t$ the blanket is $\{q_{t-1}, q_{t+1}, o_t\}$; for $o_t$ it is
just $\{q_t\}$.

Three questions: given $q_2$, is $q_3$ independent of $q_1$? (For gene finding: if you know that nucleotide 2 is in an
exon, whether nucleotide 3 is exon or intron does not depend on how nucleotide 1 is labelled.) Given $q_2$, is $q_T$
independent of $q_1$? Given $q_3$, is $o_3$ independent of $o_1$? (All yes.) Instead of trusting the graph, let us **build the entire joint table for $T = 4$**
— $2^4 \times 6^4 = 20{,}736$ numbers — and check each statement by summing and dividing (marginalising and conditioning,
Module 02).

In [ ]:
# ── 3.6 The full joint table P(q1..q4, o1..o4) ──
def full_joint_table(T, pi, trans, emit) -> np.ndarray:
    """Array J with J[q1,…,qT, o1,…,oT] = P(Q, O); shape (n,)*T + (m,)*T. Only for tiny T!"""
    n, m = emit.shape
    J = np.zeros((n,) * T + (m,) * T)
    for Q in itertools.product(range(n), repeat=T):
        p_path = pi[Q[0]] * np.prod([trans[Q[t - 1], Q[t]] for t in range(1, T)])
        emis = emit[Q[0]]
        for t in range(1, T):                       # outer product of the T emission vectors
            emis = np.multiply.outer(emis, emit[Q[t]])
        J[Q] = p_path * emis
    return J


J4 = full_joint_table(4, pi, trans, emit)
print("shape", J4.shape, "| number of entries", J4.size, "| sum =", round(J4.sum(), 12))
# axis order: 0..3 = q1..q4, 4..7 = o1..o4

In [ ]:
# ── 3.7 Verify the independence statements ──
def cond(table, axis_target):
    """Normalise a (marginal) table so that it sums to 1 along `axis_target` → a conditional distribution."""
    return table / table.sum(axis=axis_target, keepdims=True)

M = J4.sum(axis=(3, 4, 5, 6, 7))                           # P(q1, q2, q3)
d1 = np.abs(cond(M, 2) - cond(M.sum(0), 1)[None]).max()    # P(q3|q1,q2) vs P(q3|q2)
M = J4.sum(axis=(2, 4, 5, 6, 7))                           # P(q1, q2, q4)
d2 = np.abs(cond(M, 2) - cond(M.sum(0), 1)[None]).max()    # P(q4|q1,q2) vs P(q4|q2)
M = J4.sum(axis=(0, 1, 3, 5, 7))                           # P(q3, o1, o3)
d3 = np.abs(cond(M, 2) - emit[:, None, :]).max()           # P(o3|q3,o1) vs P(o3|q3)
M = J4.sum(axis=(3, 4, 6, 7))                              # P(q1, q2, q3, o2): q2's Markov blanket + q2
blanket = cond(M, 1)[:, :, :, None, None, :, None, None]   # P(q2 | q1, q3, o2), broadcast to the full shape
d4 = np.abs(cond(J4, 1) - blanket).max()                   # vs P(q2 | everything else)
print(f"max |P(q3|q1,q2) − P(q3|q2)|             = {d1:.1e}   (given q2, q3 ⊥ q1)")
print(f"max |P(q4|q1,q2) − P(q4|q2)|             = {d2:.1e}   (given q2, q_T ⊥ q1 for T = 4)")
print(f"max |P(o3|q3,o1) − P(o3|q3)|             = {d3:.1e}   (given q3, o3 ⊥ o1)")
print(f"max |P(q2|all others) − P(q2|q1,q3,o2)|  = {d4:.1e}   (Markov blanket of q2)")

# …but without conditioning on the hidden states, things ARE dependent:
Mo = J4.sum(axis=(0, 1, 2, 3, 5, 7))                       # P(o1, o3)
p_o3 = Mo.sum(0)
print(f"\nP(o3 = 6)            = {p_o3[5]:.4f}")
print(f"P(o3 = 6 | o1 = 1)   = {cond(Mo, 1)[0, 5]:.4f}")
print(f"P(o3 = 6 | o1 = 6)   = {cond(Mo, 1)[5, 5]:.4f}")
Mq = J4.sum(axis=(1, 3, 4, 5, 6, 7))                       # P(q1, q3)
print(f"P(q3 = B | q1 = A) = {cond(Mq, 1)[0, 1]:.3f}  vs  P(q3 = B | q1 = B) = {cond(Mq, 1)[1, 1]:.3f}")

🔍 **What just happened?** The four conditional-independence statements hold to machine precision (differences of order
$10^{-16}$, pure rounding). But the last lines show the other half of the story: **the observations are not independent
of one another.** Overall, $P(o_3 = 6) = 0.186$; after a 1 at roll 1 it drops to 0.173, after a 6 it rises to 0.205,
because a 6 hints that die B was in use and die B tends to stay. Likewise $q_3$ depends on $q_1$ (0.320 vs 0.680) unless we know $q_2$.
In Module 16's language, the hidden chain is a **common cause** of all observations: observations are dependent, but
become independent *given* the hidden states. A hidden variable that explains correlations is precisely what makes
latent-variable models useful.

✏️ **Your turn.** Check the statement "given $q_2$, $o_3$ is independent of $o_1$" (it should hold — why?) and the
statement "given $o_2$, $o_3$ is independent of $o_1$" (it should *fail* — why? which path in the graph stays open?).
Use `J4.sum(axis=…)` and `cond` as above.

### 📘 Lecture notes — the HMM: definition, joint probability, the two-dice example

**Random variables.** States $q_1,\dots,q_T$ (hidden, latent) take values in $S=\{s_1,\dots,s_n\}$ — e.g.
$\{I,E,B\}$ for gene finding; outputs $o_1,\dots,o_T$ (observed) take values in $\Sigma=\{\sigma_1,\dots,\sigma_m\}$ —
e.g. $\{A,C,G,T\}$.

**Conditional probabilities** (one parent per variable, except $q_1$): initial state probability
$P(q_1=s_i)=\pi_i$ (the probability that we start in $s_i$); transition probability
$a_{ij}=P(q_t=s_j\mid q_{t-1}=s_i)$, an $n\times n$ matrix shared by all $t$; emission probability
$b_i(\sigma)=P(o_t=\sigma\mid q_t=s_i)$, an $n\times m$ matrix shared by all $t$.

**Joint probability** (Bayesian-network factorisation):

$$P(Q,O)=P(q_1)\,P(o_1\mid q_1)\prod_{t=2}^{T}P(q_t\mid q_{t-1})\,P(o_t\mid q_t).$$

*In words:* initial state probability × emission, then for each step a transition × an emission.

**Two-dice example.** $\pi_A=0.7$, $\pi_B=0.3$; transitions $A\to A=B\to B=0.8$, $A\to B=B\to A=0.2$; emissions
$P(v\mid A)=(.3,.2,.2,.1,.1,.1)$, $P(v\mid B)=(.1,.1,.1,.2,.2,.3)$ for $v=1,\dots,6$. Drawn either as a Bayesian
network (a chain $q_1\to q_2\to\dots$ with $q_t\to o_t$) or as a state-transition diagram (two nodes A, B with
self-loops 0.8 and arrows 0.2, start arrows 0.7 and 0.3). Story example: states `A A A B B B B B B A A`, rolls
`1 6 2 1 4 2 1 1 1 2 5`, $P(Q,O)\approx1.1\times10^{-12}$ — and "all A" explains the same rolls about 1,300 times better.

**Markov property.** $q_{t+1}$ is independent of the earlier states given $q_t$:
$P(q_{t+1}=s_i\mid q_t=s_j)=P(q_{t+1}=s_i\mid q_t=s_j,q_{t-1}=s_k,\dots)$ for every earlier history.
**Markov blanket** (parents, children, co-parents of children): $\{q_{t-1},q_{t+1},o_t\}$ for $q_t$, $\{q_t\}$ for
$o_t$. Given $q_2$: $q_3\perp q_1$ and $q_T\perp q_1$; given $q_3$: $o_3\perp o_1$. But *without* the states the
observations are dependent.

**Parameter count.** $(n-1)+n(n-1)+n(m-1)$ — 13 for the dice, whatever $T$ — instead of $(nm)^T-1$ for a full table.

> 🎓 **Exam tip.** Know the factorisation by heart and be able to say which variables are in the Markov blanket of $q_t$ and of $o_t$. A classic trap: "the observations of an HMM are independent" — false; they are independent only *given* the states.

---
## 4 · Three questions — and why brute force fails

> 📍 **Where we are.** We can compute $P(Q,O)$ for any given path. But in practice we observe $O$ and want to learn about
> the unobserved $Q$: can we infer the unobserved from the observed — for example, what is the probability of candidate
> intron/exon/intergenic labels given the nucleotide sequence?

Given the results of the forward–backward algorithm (compute the forward variables, then the backward variables), we
can perform three **inference tasks**:

1. $P(Q \mid O)$ — the probability of one *specific* candidate path given the observations (Section 7);
2. $P(q_t = s_i \mid O)$ — the probability that position $t$ has label $s_i$, whatever the other labels are (Section 8);
3. $Q^* = \arg\max_Q P(Q \mid O)$ — the single most probable path (Section 9).

All three involve, explicitly or implicitly, the **likelihood** of the observations,

$$P(O) = \sum_{\text{all paths } Q} P(Q, O),$$

*In words:* the probability of the observed sequence is the sum of the joint probabilities over every way the hidden
states could have gone (the law of total probability). The obvious way to compute it is to list all paths. With $n$
states and $T$ positions there are $n^T$ of them.

In [ ]:
# ── 4.1 Brute force: enumerate every path ──
def brute_force_paths(obs, pi, trans, emit):
    """Return (paths, joint) where paths is an (n^T × T) array of all state paths and joint[k] = P(paths[k], O)."""
    n, T = len(pi), len(obs)
    paths = np.array(list(itertools.product(range(n), repeat=T)))
    joint = pi[paths[:, 0]] * emit[paths[:, 0], obs[0]]
    for t in range(1, T):                                  # vectorised over all paths at once
        joint = joint * trans[paths[:, t - 1], paths[:, t]] * emit[paths[:, t], obs[t]]
    return paths, joint


O_example = to_obs("1 2 2 1 1 2 1 1 1 2 2")                  # the example sequence of Section 9
t0 = time.perf_counter()
paths11, joint11 = brute_force_paths(O_example, pi, trans, emit)
print(f"T = {len(O_example)}: {len(paths11)} paths, P(O) = {joint11.sum():.6g}  ({time.perf_counter() - t0:.3f} s)")

In [ ]:
# ── 4.2 How the work grows: n^T paths vs n²T for the forward algorithm ──
Ts = np.arange(1, 41)
n = 2
ops_brute = n ** Ts.astype(float) * (2 * Ts)     # each path: ≈ 2T multiplications (plus one addition)
ops_forward = 2 * n ** 2 * Ts                     # forward algorithm (Section 5): ≈ 2n² operations per position

fig, ax = plt.subplots(figsize=(6, 3.2))
ax.semilogy(Ts, ops_brute, color="tab:red", label="brute force: $n^T$ paths × 2T")
ax.semilogy(Ts, ops_forward, color="tab:blue", label="forward algorithm: ≈ $2n^2T$")
ax.set_xlabel("sequence length T (n = 2 states)"); ax.set_ylabel("arithmetic operations (log scale)")
ax.set_title("exponential vs linear growth of the work"); ax.legend()
plt.tight_layout(); plt.show()

for T in [11, 50, 100]:
    years = 2.0 ** T / 1e9 / 3.15e7
    print(f"T = {T:3d}: 2^T = {2.0 ** T:.3g} paths → {years:.3g} years at a billion paths per second")

🔍 **What just happened?** Enumerating the 2,048 paths of an 11-roll sequence is instant, but every extra roll doubles
the work. At $T = 100$ there are $1.3\times 10^{30}$ paths — about $4\times10^{13}$ years at a billion paths per second,
some 3,000 times the age of the universe. A human gene spans tens of thousands of bases and a gene finder has $n \ge 3$
labels. Brute force is hopeless; yet the forward algorithm (blue line) grows only *linearly* with $T$. The trick is the
same as in variable elimination ([Module 17](17_bayesian_networks_inference_learning.ipynb)): **reuse partial sums** instead of recomputing them for every path.

✏️ **Your turn.** Change `n = 2` to `n = 3` (a three-label gene finder) in cell 4.2. For which $T$ does brute force
first need more than a million operations? And how many operations does the forward algorithm need for a 3-state model
on a 10,000-base sequence?

---
## 5 · The forward algorithm

> 📍 **Where we are.** We need $P(O) = \sum_Q P(Q, O)$ without listing the $n^T$ paths. The forward algorithm does it in
> $O(n^2 T)$ by sweeping once from left to right.

### 5.1 The forward variable

Define, for each position $t$ and each state $s_i$,

$$\alpha_t(i) = P(o_1, o_2, \dots, o_t,\; q_t = s_i).$$

*In words:* $\alpha_t(i)$ is the probability of having seen the first $t$ observations **and** being in state $s_i$ at
time $t$ — summed over all the ways the earlier states could have gone. It is a "running total" over all path
prefixes that end in $s_i$.

If we had $\alpha_T(i)$ for every $i$, we would be done, because by the law of total probability

$$P(O) = \sum_{i=1}^{n} P(o_1,\dots,o_T, q_T = s_i) = \sum_{i=1}^n \alpha_T(i).$$

*In words:* add up the running totals of the last column, one per possible final state.

🧮 **Math refresher — the three tools of every HMM derivation.** We use only three rules, and we label each step
with one of them ([Module 02](02_probability_for_biologists.ipynb), Module 16):

1. **Law of total probability** (sum over a variable you do not know): $P(X) = \sum_{y} P(X, Y = y)$.
2. **Product rule** (written "$P(A,B) = P(A\mid B)P(B)$"): a joint probability = a conditional × a marginal.
   It also works with extra conditions on every term: $P(A, B \mid C) = P(A \mid B, C)\,P(B \mid C)$.
3. **Markov property / conditional independence read from the graph**: in a conditional probability, you may delete
   conditioning variables that are separated from the target by the ones that remain. In an HMM: *given $q_t$, everything
   that happens after $t$ is independent of everything up to $t$*, and *given $q_t$, $o_t$ is independent of all else*.

### 5.2 Initialisation

$$\alpha_1(i) = P(o_1, q_1 = s_i) = P(o_1 \mid q_1 = s_i)\,P(q_1 = s_i) = b_i(o_1)\,\pi_i.$$

*In words:* the probability of starting in $s_i$ and emitting the first observation is "start probability × emission
probability" (product rule).

### 5.3 The recursion, derived line by line

Suppose we know $\alpha_t(j)$ for every $j$. Then

$$
\begin{aligned}
\alpha_{t+1}(i) &= P(o_1,\dots,o_t,o_{t+1},\; q_{t+1}=s_i) &&\text{[definition]}\\
&= \sum_{j=1}^{n} P(o_1,\dots,o_{t+1},\; q_t = s_j,\; q_{t+1}=s_i) &&\text{[law of total probability, over } q_t]\\
&= \sum_{j=1}^{n} P(o_{t+1}, q_{t+1}=s_i \mid o_1,\dots,o_t, q_t=s_j)\; P(o_1,\dots,o_t, q_t=s_j) &&\text{[product rule]}\\
&= \sum_{j=1}^{n} P(o_{t+1}, q_{t+1}=s_i \mid q_t=s_j)\; \alpha_t(j) &&\text{[Markov property + definition of } \alpha_t]\\
&= \sum_{j=1}^{n} P(o_{t+1}\mid q_{t+1}=s_i, q_t=s_j)\; P(q_{t+1}=s_i\mid q_t=s_j)\; \alpha_t(j) &&\text{[product rule]}\\
&= \sum_{j=1}^{n} P(o_{t+1}\mid q_{t+1}=s_i)\; P(q_{t+1}=s_i\mid q_t=s_j)\; \alpha_t(j) &&\text{[Markov property]}\\
&= b_i(o_{t+1}) \sum_{j=1}^{n} a_{ji}\, \alpha_t(j).
\end{aligned}
$$

*In words, line by line:* (1) write down what we want; (2) we do not know the previous state, so sum over its $n$
possible values; (3) split the joint into "what happens at $t+1$, given the past" × "the past"; (4) given the state at
$t$, what happens at $t+1$ does not care about the earlier observations — and "the past" is exactly $\alpha_t(j)$;
(5) split "move to $s_i$ and emit $o_{t+1}$" into "move" × "emit given where we are and came from"; (6) the emission
depends only on the current state. The last line pulls the emission (which does not depend on $j$) out of the sum.

**Summary of the forward algorithm:**

$$\alpha_1(i) = \pi_i\, b_i(o_1), \qquad \alpha_{t+1}(i) = b_i(o_{t+1}) \sum_{j=1}^n \alpha_t(j)\, a_{ji}, \qquad P(O) = \sum_i \alpha_T(i).$$

*In words:* start with "initial × emission"; at each step, **collect** probability from every previous state through the
transition arrows into $s_i$, then multiply by the probability that $s_i$ emits the new observation; at the end, add up.

🧮 **Vector form (Module 03).** If $\boldsymbol\alpha_t$ is the row vector $(\alpha_t(1),\dots,\alpha_t(n))$, then
$\sum_j \alpha_t(j) a_{ji}$ is the $i$-th entry of the vector–matrix product $\boldsymbol\alpha_t\, \mathbf A$. In NumPy:
`alpha[t-1] @ trans`. Multiplying element-wise by the column of emission probabilities for $o_{t+1}$ finishes the step.

### 5.4 A worked example: O = (2, 3, 6)

We observe the rolls $O=(2,3,6)$ and use the two-dice parameters. The next cell defines `forward`. Then, before running
it, we do the calculation by hand:

* $\alpha_1(A)=P(o_1=2,q_1=A)=P(o_1=2\mid q_1=A)\,\pi_A=.2\times.7=.14$, and $\alpha_1(B)=.1\times.3=.03$;
* $\alpha_2(A)=\sum_{j=A,B}P(o_2=3\mid q_2=A)\,P(q_2=A\mid q_1=j)\,\alpha_1(j)=.2\times.8\times.14+.2\times.2\times.03=.0236$,
  and likewise $\alpha_2(B)=.0052$;
* and then the third column, $\alpha_3(A)$ and $\alpha_3(B)$ — try it yourself before reading the output.

In [ ]:
# ── 5.1 The forward algorithm ──
def forward(obs, pi, trans, emit) -> np.ndarray:
    """Forward table: alpha[t, i] = P(o_1..o_{t+1}, q_{t+1} = s_i)  (row t of the array = time t+1 of the formulas).

    obs: 0-based observation indices; pi: (n,), trans: (n, n) with rows = 'from'; emit: (n, m).
    """
    T, n = len(obs), len(pi)
    alpha = np.zeros((T, n))
    alpha[0] = pi * emit[:, obs[0]]                         # initialisation: π_i · b_i(o_1)
    for t in range(1, T):
        # recursion: collect Σ_j α_t(j) a_ji for every i (vector–matrix product), then emit o_{t+1}
        alpha[t] = emit[:, obs[t]] * (alpha[t - 1] @ trans)
    return alpha

In [ ]:
# ── 5.2 By hand ──
bA = dict(zip(FACES, emit[0])); bB = dict(zip(FACES, emit[1]))
a1A = bA[2] * 0.7;  a1B = bB[2] * 0.3
print(f"α1(A) = P(o1=2|A)·π_A = {bA[2]} × 0.7 = {a1A:.4g}")
print(f"α1(B) = P(o1=2|B)·π_B = {bB[2]} × 0.3 = {a1B:.4g}")
a2A = bA[3] * (0.8 * a1A + 0.2 * a1B)
a2B = bB[3] * (0.2 * a1A + 0.8 * a1B)
print(f"α2(A) = P(3|A)·[a_AA α1(A) + a_BA α1(B)] = {bA[3]} × (0.8×{a1A:.2f} + 0.2×{a1B:.2f}) = {a2A:.4g}")
print(f"α2(B) = P(3|B)·[a_AB α1(A) + a_BB α1(B)] = {bB[3]} × (0.2×{a1A:.2f} + 0.8×{a1B:.2f}) = {a2B:.4g}")
a3A = bA[6] * (0.8 * a2A + 0.2 * a2B)
a3B = bB[6] * (0.2 * a2A + 0.8 * a2B)
print(f"α3(A) = {bA[6]} × (0.8×{a2A:.4f} + 0.2×{a2B:.4f}) = {a3A:.6g}")
print(f"α3(B) = {bB[6]} × (0.2×{a2A:.4f} + 0.8×{a2B:.4f}) = {a3B:.6g}")
print(f"P(O = 2,3,6) = α3(A) + α3(B) = {a3A + a3B:.6g}")

In [ ]:
# ── 5.3 The same with the function: the n × T table, printed and drawn as a trellis ──
O236 = to_obs("2 3 6")
alpha236 = forward(O236, pi, trans, emit)
table = pd.DataFrame(alpha236.T, index=[f"state {s}" for s in STATES],
                     columns=[f"t={t + 1} (o={o + 1})" for t, o in enumerate(O236)])
print(table.round(6))
print("P(O) =", alpha236[-1].sum())


# Picture the same table as a trellis: one node per (time, state), lines = possible transitions
def draw_trellis(ax, values, obs, title, fmt="{:.4g}", names=STATES, colors=DIE_COLORS):
    """Draw an n × T trellis: one node per (time, state), annotated with values[t, i]; grey lines = transitions."""
    T, n = values.shape
    for t in range(T - 1):
        for i in range(n):
            for j in range(n):
                ax.plot([t, t + 1], [n - 1 - i, n - 1 - j], color="0.8", lw=1, zorder=1)
    for t in range(T):
        for i in range(n):
            ax.scatter(t, n - 1 - i, s=1900, color=colors[i], alpha=0.3, edgecolor="0.3", zorder=2)
            ax.text(t, n - 1 - i, fmt.format(values[t, i]), ha="center", va="center", fontsize=8, zorder=3,
                    bbox=dict(fc="white", ec="none", alpha=0.75, pad=0.6))
    ax.set_xticks(range(T)); ax.set_xticklabels([f"t={t + 1}\no={obs[t] + 1}" for t in range(T)])
    ax.set_yticks(range(n)); ax.set_yticklabels([f"die {s}" for s in names[::-1]])
    ax.set_xlim(-0.5, T - 0.5); ax.set_ylim(-0.6, n - 0.4); ax.set_title(title)
    for side in ("left", "bottom"):
        ax.spines[side].set_visible(False)
    ax.tick_params(length=0)


fig, ax = plt.subplots(figsize=(6, 2.6))
draw_trellis(ax, alpha236, O236, r"forward table $\alpha_t(i)$ for O = (2, 3, 6); P(O) = sum of last column")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Hand calculation, function and trellis agree: $\alpha_1(A)=.14$,
$\alpha_1(B)=.03$, $\alpha_2(A)=.0236$, $\alpha_2(B)=.0052$. The third column is
$\alpha_3(B) = 0.3\times(0.2\times0.0236 + 0.8\times0.0052) = 0.002664$, and $\alpha_3(A)=0.001992$, so
$P(O)=0.004656$. Notice how the evidence shifts: after "2" and "3" die A leads by a factor of about 4.5, but the final
"6" (three times more likely under B) flips the last column in favour of B.

Is $P(O) = 0.004656$ right? There are only $2^3 = 8$ paths, so let us list them all.

In [ ]:
# ── 5.5 Brute-force check over all 8 paths ──
paths3, joint3 = brute_force_paths(O236, pi, trans, emit)
df = pd.DataFrame({"path": ["".join(STATES[s] for s in p) for p in paths3], "P(Q, O)": joint3})
df["ends in"] = df["path"].str[-1]
print(df.to_string(index=False))
print(f"\nsum over all paths            = {joint3.sum():.6g}   (forward: {alpha236[-1].sum():.6g})")
print(f"sum over paths ending in A    = {df.loc[df['ends in'] == 'A', 'P(Q, O)'].sum():.6g}   (α3(A) = {alpha236[-1, 0]:.6g})")
print(f"sum over paths ending in B    = {df.loc[df['ends in'] == 'B', 'P(Q, O)'].sum():.6g}   (α3(B) = {alpha236[-1, 1]:.6g})")

In [ ]:
# ── 5.6 Forward vs brute force on random sequences ──
rng = np.random.default_rng(SEED + 5)
for trial in range(5):
    _, obs_r = sample_hmm(10, pi, trans, emit, rng)
    pf = forward(obs_r, pi, trans, emit)[-1].sum()
    pb = brute_force_paths(obs_r, pi, trans, emit)[1].sum()
    print(f"faces {''.join(str(o + 1) for o in obs_r)}:  forward {pf:.6e}   brute force (1,024 paths) {pb:.6e}   "
          f"relative difference {abs(pf - pb) / pb:.1e}")

🔍 **What just happened?** The forward algorithm reproduces brute force exactly — not approximately: the relative
differences are at the level of floating-point rounding. Each $\alpha_3(i)$ is the sum of the joint probabilities of the
four paths ending in state $i$. That is the whole idea: **$\alpha_t(i)$ is the total over all path prefixes ending in
$s_i$**, and because the future depends on the past only through the current state, these totals can be extended one
step at a time without ever separating the individual paths again.

**Complexity.** Each of the $T$ columns needs, for each of the $n$ states, a sum over $n$ predecessors:
$O(n^2 T)$ operations instead of $O(n^T T)$.

✏️ **Your turn.** By hand, compute $\alpha_1$ and $\alpha_2$ for $O=(6, 6)$ and the resulting $P(O)$; then check with
`forward(to_obs("6 6"), pi, trans, emit)`. Is two sixes in a row more or less likely than "2, 3"? Why?

### 📘 Lecture notes — inference tasks and the forward algorithm

**Three inference tasks** (each uses the forward and/or backward variables): $P(Q\mid O)$ for one candidate path;
$P(q_t=s_i\mid O)$ for one position; the most probable path $\arg\max_Q P(Q\mid O)$. All need $P(O)=\sum_Q P(Q,O)$, a sum
over $n^T$ paths.

**Forward variable.** $\alpha_t(i)=P(o_1,\dots,o_t,\;q_t=s_i)$ — the past observations *and* the current state.

**Initialisation** (emission × initial state): $\alpha_1(i)=P(o_1\mid q_1=s_i)P(q_1=s_i)=b_i(o_1)\,\pi_i$.

**Recursion** (law of total probability over $q_t$; product rule $P(A,B)=P(A\mid B)P(B)$; Markov property + the
definition of $\alpha_t$; product rule; Markov property):

$$\alpha_{t+1}(i)=b_i(o_{t+1})\sum_{j=1}^n\alpha_t(j)\,a_{ji},\qquad P(O)=\sum_i\alpha_T(i).$$

*In words:* collect probability from every previous state through the arrows into $s_i$ (transition), then emit
the new observation (emission); fill the $n\times T$ table column by column. **Complexity** $O(n^2T)$ instead of
$O(n^T T)$.

**Worked example** $O=(2,3,6)$: $\alpha_1(A)=.2\times.7=.14$, $\alpha_1(B)=.1\times.3=.03$;
$\alpha_2(A)=.2\times(.8\times.14+.2\times.03)=.0236$, $\alpha_2(B)=.1\times(.2\times.14+.8\times.03)=.0052$;
$\alpha_3(A)=.1\times(.8\times.0236+.2\times.0052)=.001992$, $\alpha_3(B)=.3\times(.2\times.0236+.8\times.0052)=.002664$;
$P(O)=.004656$ — the same as summing $P(Q,O)$ over all 8 paths.

> 🎓 **Exam tip.** Be able to reproduce the derivation with the reason for each equality — the Markov step that drops $o_1,\dots,o_t$ from the condition is the one students most often cannot justify. In numerical questions, the classic slip is to use $a_{ij}$ instead of $a_{ji}$: the sum runs over *previous* states $j$ with arrows *into* $i$.

---
## 6 · The backward algorithm

> 📍 **Where we are.** The forward variable summarises the past. To ask about a state in the *middle* of the sequence
> (Section 8) we also need a summary of the **future**. The backward algorithm sweeps from right to left.

### 6.1 The backward variable

$$\beta_t(i) = P(o_{t+1}, o_{t+2}, \dots, o_T \mid q_t = s_i).$$

*In words:* if we were in state $s_i$ at time $t$, how probable would all the **later** observations be? Note the
difference from $\alpha$: this is a *conditional* probability and it is about the future, not the past.

**Initialisation.** At $t = T$ there are no later observations ($o_{T+1}$ does not exist). The probability of "nothing" is 1, so by
convention

$$\beta_T(i) = 1 \quad \text{for every } i.$$

*In words:* an empty set of future observations is certain to be "observed".

### 6.2 The recursion, derived line by line

$$
\begin{aligned}
\beta_t(i) &= P(o_{t+1},\dots,o_T \mid q_t = s_i) &&\text{[definition]}\\
&= \sum_{j=1}^n P(o_{t+1},\dots,o_T,\; q_{t+1}=s_j \mid q_t=s_i) &&\text{[law of total probability, over } q_{t+1}]\\
&= \sum_{j=1}^n P(o_{t+1},\dots,o_T \mid q_{t+1}=s_j, q_t=s_i)\; P(q_{t+1}=s_j\mid q_t=s_i) &&\text{[product rule]}\\
&= \sum_{j=1}^n P(o_{t+1},\dots,o_T \mid q_{t+1}=s_j)\; P(q_{t+1}=s_j\mid q_t=s_i) &&\text{[Markov property]}\\
&= \sum_{j=1}^n P(o_{t+2},\dots,o_T \mid o_{t+1}, q_{t+1}=s_j)\; P(o_{t+1}\mid q_{t+1}=s_j)\; P(q_{t+1}=s_j\mid q_t=s_i) &&\text{[product rule]}\\
&= \sum_{j=1}^n P(o_{t+2},\dots,o_T \mid q_{t+1}=s_j)\; P(o_{t+1}\mid q_{t+1}=s_j)\; P(q_{t+1}=s_j\mid q_t=s_i) &&\text{[Markov property]}\\
&= \sum_{j=1}^n \beta_{t+1}(j)\; b_j(o_{t+1})\; a_{ij} &&\text{[definition of } \beta_{t+1}].
\end{aligned}
$$

*In words, line by line:* (2) we do not know the next state, so sum over it; (3) split into "future observations given
both states" × "transition"; (4) once $q_{t+1}$ is known, $q_t$ adds nothing about observations from $t+1$ on; (5) peel off
the first future observation $o_{t+1}$ with the product rule; (6) given $q_{t+1}$, the observations after $t+1$ do not
care about $o_{t+1}$; (7) recognise $\beta_{t+1}(j)$.

**Summary of the backward algorithm:** $\beta_T(i) = 1$ and $\beta_t(i) = \sum_j a_{ij}\, b_j(o_{t+1})\, \beta_{t+1}(j)$.
*In words:* to look ahead from $s_i$, **distribute** probability over every next state $s_j$ (transition), let $s_j$ emit
the next observation (emission), and multiply by how well $s_j$ explains everything after that. In vector form,
$\boldsymbol\beta_t = \mathbf A\,(\mathbf b(o_{t+1}) \odot \boldsymbol\beta_{t+1})$ — `trans @ (emit[:, obs[t+1]] * beta[t+1])`.

⚠️ **Which factor is which?** In the last line, $P(q_{t+1}=s_j\mid q_t=s_i)=a_{ij}$ is the **transition** and
$P(o_{t+1}\mid q_{t+1}=s_j)=b_j(o_{t+1})$ the **emission** — and the emission belongs to the *next* position $t+1$, not to
$t$. Mixing the two up (or emitting $o_t$ instead of $o_{t+1}$) is the most common bug in backward code.

### 6.3 Forward × backward = P(O), at every position

Why should $\sum_i \alpha_t(i)\beta_t(i)$ equal $P(O)$ for **every** $t$, not just at the ends? Because

$$\alpha_t(i)\,\beta_t(i) = P(o_1..o_t, q_t=s_i)\; P(o_{t+1}..o_T \mid q_t = s_i) = P(o_1..o_t, q_t=s_i)\; P(o_{t+1}..o_T \mid o_1..o_t, q_t = s_i) = P(O, q_t = s_i).$$

*In words:* "the past ending in $s_i$" × "the future given $s_i$" is the joint probability of the whole sequence and
$q_t = s_i$ — the middle step is the Markov property (given $q_t$, the future ignores past observations), the last
is the product rule. Summing over $i$ (law of total probability) gives $P(O)$. This identity is the best unit test for
an HMM implementation.

The next two cells implement the backward pass, reproduce the worked example by hand, and run this unit test.

In [ ]:
# ── 6.1 The backward algorithm ──
def backward(obs, trans, emit) -> np.ndarray:
    """Backward table: beta[t, i] = P(o_{t+2}..o_T | q_{t+1} = s_i)  (row t = time t+1 of the formulas); last row = 1."""
    T, n = len(obs), trans.shape[0]
    beta = np.ones((T, n))                                   # initialisation: β_T(i) = 1
    for t in range(T - 2, -1, -1):                           # sweep right → left
        beta[t] = trans @ (emit[:, obs[t + 1]] * beta[t + 1])   # Σ_j a_ij b_j(o_{t+1}) β_{t+1}(j)
    return beta


beta236 = backward(O236, trans, emit)
print(pd.DataFrame(beta236.T, index=[f"state {s}" for s in STATES],
                   columns=[f"t={t + 1} (o={o + 1})" for t, o in enumerate(O236)]).round(6))
print(f"\nby hand: β2(A) = a_AA·b_A(6)·1 + a_AB·b_B(6)·1 = 0.8×0.1 + 0.2×0.3 = {0.8 * 0.1 + 0.2 * 0.3:.4g}")
print(f"         β2(B) = a_BA·b_A(6)·1 + a_BB·b_B(6)·1 = 0.2×0.1 + 0.8×0.3 = {0.2 * 0.1 + 0.8 * 0.3:.4g}")
p_from_beta = np.sum(pi * emit[:, O236[0]] * beta236[0])
print(f"P(O) from the backward table: Σ_i π_i b_i(o_1) β_1(i) = {p_from_beta:.6g}")

In [ ]:
# ── 6.2 Unit test: Σ_i α_t(i) β_t(i) = P(O) for every t ──
print("O = (2, 3, 6):  Σ_i α_t(i)β_t(i) =", np.round((alpha236 * beta236).sum(axis=1), 8))
rng = np.random.default_rng(SEED + 6)
_, obs50 = sample_hmm(50, pi, trans, emit, rng)
a50, b50 = forward(obs50, pi, trans, emit), backward(obs50, trans, emit)
col = (a50 * b50).sum(axis=1)
print(f"random 50-roll sequence: P(O) = {a50[-1].sum():.5e}; Σ_i α_t β_t ranges over "
      f"[{col.min():.5e}, {col.max():.5e}] for t = 1..50 (max relative spread {np.ptp(col) / col.mean():.1e})")

🔍 **What just happened?** The backward table matches the hand calculation ($\beta_2 = (0.14, 0.26)$,
$\beta_1 = (0.0276, 0.0264)$), and the forward–backward product gives the same $P(O) = 0.004656$ in every column — and the
same for a random 50-roll sequence, up to rounding. Two independent sweeps, one identity: our code is almost certainly right.

✏️ **Your turn.** Why is $\beta_2(B) > \beta_2(A)$ for this sequence? (Hint: $\beta_2$ only "sees" the last observation, a 6.)
What would $\beta_2$ be if the last roll were a 1?

---
## 7 · The probability of one specific path, P(Q | O)

> 📍 **Where we are.** Inference task 1: given the observations, how probable is a particular candidate
> labelling — say, a proposed gene structure?

Bayes' rule (Module 02) gives

$$P(Q \mid O) = \frac{P(O \mid Q)\, P(Q)}{P(O)}.$$

*In words:* the posterior probability of a path is how well it explains the data, times how plausible it was before
seeing the data, divided by the total probability of the data. The pieces:

* $P(O\mid Q) = P(o_1\mid q_1)\cdots P(o_T\mid q_T)$ — easy, $O(T)$;
* $P(Q) = P(q_1)P(q_2\mid q_1)\cdots P(q_T\mid q_{T-1})$ — easy, $O(T)$;
* $P(O)$ — hard, a sum over all paths; but $P(O)=\sum_i P(o_1,\dots,o_T,q_T=s_i)=\sum_i \alpha_T(i)$, so the forward
  algorithm makes it $O(n^2T)$.

In [ ]:
# ── 7.1 P(Q | O) for every path of O = (2, 3, 6) and for the story example of Section 3.2 ──
def path_posterior(states, obs, pi, trans, emit) -> float:
    """P(Q | O) = P(O|Q) P(Q) / P(O), with P(O) from the forward algorithm."""
    p_O_given_Q = np.prod(emit[states, obs])                                          # O(T)
    p_Q = pi[states[0]] * np.prod(trans[states[:-1], states[1:]])                     # O(T)
    p_O = forward(obs, pi, trans, emit)[-1].sum()                                      # O(n² T)
    return p_O_given_Q * p_Q / p_O


post3 = {"".join(STATES[s] for s in p): path_posterior(np.array(p), O236, pi, trans, emit) for p in paths3}
for path, pr in sorted(post3.items(), key=lambda kv: -kv[1]):
    print(f"P(Q = {path} | O = 2,3,6) = {pr:.4f}")
print("sum over the 8 paths:", round(sum(post3.values()), 12))
print(f"\nstory example: P(Q = AAABBBBBBAA | O) = {path_posterior(Q21, O21, pi, trans, emit):.3g}")

🔍 **What just happened?** For $O=(2,3,6)$ the eight posterior path probabilities sum to 1, and the best path is AAA with
probability 0.385, followed by AAB (0.289). For the 11-roll story example of Section 3.2 the path that generated the data has a
posterior probability of only about $1.6\times10^{-4}$: the posterior is spread over thousands of paths, none of
which is individually very likely. For long sequences the probability of *any* single complete path is minuscule. That
is why we usually ask more robust questions — "how likely is each **position**'s label?" (Section 8) or "which single
path is *most* probable?" (Section 9).

✏️ **Your turn.** Compute $P(Q\mid O)$ for $Q$ = BBB and $O = (6, 6, 6)$, first with `path_posterior`, then by hand from
the forward table (`forward(to_obs("6 6 6"), pi, trans, emit)`). Is BBB now the most probable of the 8 paths? Then
compute $P(\text{AAA}\mid O = 1, 1, 1)$. Which "story" is more convincing, and why — given that the emission ratios
$b_B(6)/b_A(6)$ and $b_A(1)/b_B(1)$ are both exactly 3? (Hint: look at $\pi$.)

### 📘 Lecture notes — the backward algorithm and P(Q | O)

**Backward variable.** $\beta_t(i)=P(o_{t+1},\dots,o_T\mid q_t=s_i)$ — the future observations given the current
state (a conditional probability, unlike $\alpha$).

**Initialisation.** $\beta_T(i)=1$ — by definition, because $o_{T+1}$ does not exist.

**Recursion** (law of total probability over $q_{t+1}$; product rule; Markov; product rule; Markov; recursion):

$$\beta_t(i)=\sum_{j=1}^n P(q_{t+1}=s_j\mid q_t=s_i)\,P(o_{t+1}\mid q_{t+1}=s_j)\,\beta_{t+1}(j)=\sum_j a_{ij}\,b_j(o_{t+1})\,\beta_{t+1}(j).$$

*In words:* distribute probability to every next state (transition), let it emit the next observation (emission),
and multiply by how well it explains the rest; fill the table from $t=T$ back to $t=1$.

**Worked example** $O=(2,3,6)$: $\beta_3=(1,1)$; $\beta_2(A)=.8\times.1+.2\times.3=.14$,
$\beta_2(B)=.2\times.1+.8\times.3=.26$; $\beta_1=(.0276,.0264)$; and $\sum_i\pi_ib_i(o_1)\beta_1(i)=.004656=P(O)$.
Unit test: $\sum_i\alpha_t(i)\beta_t(i)=P(O)$ for **every** $t$.

**P(Q | O) for one path** (Bayes' rule): $P(Q\mid O)=P(O\mid Q)P(Q)/P(O)$ with $P(O\mid Q)=\prod_tP(o_t\mid q_t)$ (easy,
$O(T)$), $P(Q)=P(q_1)\prod_tP(q_t\mid q_{t-1})$ (easy, $O(T)$) and $P(O)=\sum_i\alpha_T(i)$ (hard by enumeration;
$O(n^2T)$ by the forward algorithm). For $O=(2,3,6)$: $P(AAA\mid O)=.385$, $P(AAB\mid O)=.289$.

> 🎓 **Exam tip.** Two classic errors: forgetting the emission $b_j(o_{t+1})$ inside the backward sum (it belongs to the *next* position), and initialising $\beta_T$ with the emission probabilities instead of 1. Check yourself with $\sum_i \pi_i b_i(o_1)\beta_1(i) = \sum_i \alpha_T(i)$.

---
## 8 · Posterior decoding: the probability of each position's state

> 📍 **Where we are.** Inference task 2: $P(q_t = s_i \mid O)$ — "how likely is it that base $t$ is in an exon?",
> "how likely is it that roll $t$ was made with die B?" — whatever the other labels are.

The derivation, with the rule used at each step:

$$
\begin{aligned}
P(q_t = A \mid o_1,\dots,o_T) &= \frac{P(o_1,\dots,o_t,o_{t+1},\dots,o_T,\; q_t = A)}{P(o_1,\dots,o_T)} &&\text{[definition of conditional probability]}\\
&= \frac{P(o_1,\dots,o_t, q_t = A)\; P(o_{t+1},\dots,o_T \mid o_1,\dots,o_t, q_t = A)}{P(o_1,\dots,o_T)} &&\text{[product rule]}\\
&= \frac{P(o_1,\dots,o_t, q_t = A)\; P(o_{t+1},\dots,o_T \mid q_t = A)}{P(o_1,\dots,o_T)} &&\text{[Markov property]}\\
&= \frac{\alpha_t(A)\,\beta_t(A)}{\sum_i \alpha_t(i)\,\beta_t(i)} &&\text{[definitions; law of total probability for the denominator]}.
\end{aligned}
$$

*In words:* the posterior probability of a state at position $t$ is "everything before, ending in that state" ×
"everything after, starting from that state", normalised so the $n$ possibilities add to 1. One forward and one
backward sweep give this for **every** $t$ and **every** state at once — $O(n^2T)$ in total. Choosing the most
probable state at each position, $\hat q_t = \arg\max_i P(q_t=s_i\mid O)$, is called **posterior decoding**.

In [ ]:
# ── 8.1 Posterior state probabilities ──
def posterior_states(obs, pi, trans, emit) -> np.ndarray:
    """gamma[t, i] = P(q_{t+1} = s_i | O) = α β / Σ α β  (unscaled version; fine for T up to a few hundred)."""
    alpha, beta = forward(obs, pi, trans, emit), backward(obs, trans, emit)
    ab = alpha * beta
    return ab / ab.sum(axis=1, keepdims=True)


gamma236 = posterior_states(O236, pi, trans, emit)
print(pd.DataFrame(gamma236, columns=[f"P(q_t={s}|O)" for s in STATES],
                   index=[f"t={t + 1} (o={o + 1})" for t, o in enumerate(O236)]).round(4))
print("posterior decoding:", "".join(STATES[i] for i in gamma236.argmax(1)),
      "| most probable single path (Section 7): AAA")

# check against brute force for a random 10-roll sequence
rng = np.random.default_rng(SEED + 8)
_, obs10 = sample_hmm(10, pi, trans, emit, rng)
P10, J10 = brute_force_paths(obs10, pi, trans, emit)
gamma_bf = np.stack([[J10[P10[:, t] == i].sum() for i in range(2)] for t in range(10)]) / J10.sum()
print(f"\nrandom 10-roll sequence: max |γ(forward–backward) − γ(brute force)| = "
      f"{np.abs(posterior_states(obs10, pi, trans, emit) - gamma_bf).max():.1e}")

🔍 **What just happened?** For $O = (2, 3, 6)$: $P(q_1=A\mid O)=0.830$, $P(q_2=A\mid O)=0.710$ and $P(q_3=A\mid O)=0.428$ —
so posterior decoding labels the rolls **A, A, B**. Yet the single most probable path was **AAA** (0.385, against 0.289
for AAB). There is no contradiction: $P(q_3 = B \mid O) = 0.572$ adds up the paths AAB, ABB, BBB and BAB, while AAA is the
largest *single* path. Posterior decoding and the most probable path answer **different questions** and can disagree —
here already on a 3-roll sequence. The brute-force comparison confirms the formula ($\sim10^{-16}$ differences).

Now a realistic test: a long simulated sequence where we know the truth.

In [ ]:
# ── 8.2 Decode a long SIMULATED sequence ──
rng = np.random.default_rng(SEED + 82)
T_long = 200                       # kept ≤ a few hundred on purpose: Section 10 explains why
q_true, o_long = sample_hmm(T_long, pi, trans, emit, rng)
gamma_long = posterior_states(o_long, pi, trans, emit)
q_post = gamma_long.argmax(axis=1)
print(f"posterior decoding accuracy: {accuracy(q_true, q_post):.3f}   "
      f"(always guessing the more common die would give {max(np.mean(q_true == 0), np.mean(q_true == 1)):.3f})")

fig, ax = plt.subplots(figsize=(10, 2.8))
t_axis = np.arange(1, T_long + 1)
ax.fill_between(t_axis, 0, q_true, step="mid", color="tab:orange", alpha=0.25, label="true die B (hidden)")
ax.plot(t_axis, gamma_long[:, 1], color="tab:blue", lw=1.3, label=r"posterior $P(q_t = B \mid O)$")
ax.axhline(0.5, color="0.5", lw=0.8, ls="--")
ax.set_xlabel("roll t"); ax.set_ylabel("probability"); ax.set_ylim(-0.02, 1.02)
ax.set_title(f"SIMULATED: posterior decoding of {T_long} dice rolls (accuracy {accuracy(q_true, q_post):.2f})")
ax.legend(loc="upper right", ncol=2); plt.tight_layout(); plt.show()

🔍 **What just happened?** The posterior curve rises inside most orange (die-B) stretches and falls inside die-A
stretches, and posterior decoding labels 73.5 % of the rolls correctly (always guessing the more common die would be
right 63 % of the time). It is far from perfect, and it should be:
the two dice are not very different (die B's 6 is only three times more likely than die A's 6), so short runs are
practically invisible and the curve is uncertain (near 0.5) around many switches. The posterior is honest about this —
values near 0.5 *are* the model saying "I cannot tell".

✏️ **Your turn.** Make die B a strongly loaded die, `emit_loaded = np.array([[0.3, .2, .2, .1, .1, .1], [.04, .04, .04, .04, .04, .8]])`,
simulate and decode with it (pass it to both `sample_hmm` and `posterior_states`). How much does the accuracy improve?
What happens if you decode the loaded-die data with the *wrong* (original) emission table?

---
## 9 · The most probable path: the Viterbi algorithm

> 📍 **Where we are.** Inference task 3: one single labelling — "the words" in speech, "the set of genes" in a genome.
> Different candidate paths $Q^1, Q^2, \dots$ have different posterior probabilities $P(Q^1\mid O), P(Q^2\mid O), \dots$;
> we want the best one, $Q^* = \arg\max_Q P(Q \mid O)$.

**Step 1 — drop the denominator.** $P(O)$ does not depend on $Q$, so

$$\arg\max_Q P(Q\mid O) = \arg\max_Q \frac{P(O\mid Q)P(Q)}{P(O)} = \arg\max_Q P(O, Q).$$

*In words:* the path that is most probable *given* the data is the path with the largest *joint* probability with the
data; we never need $P(O)$.

**Step 2 — a running maximum instead of a running sum.** Define

$$\delta_t(i) = \max_{q_1,\dots,q_{t-1}} P(q_1,\dots,q_{t-1},\; q_t = s_i,\; o_1,\dots,o_t).$$

*In words:* among all path prefixes that end in state $s_i$ at time $t$, the joint probability of the **best** one.
Exactly the same reasoning as for $\alpha$ — but with "max" where the forward algorithm had "$\sum$" — gives

$$\delta_1(i) = \pi_i\, b_i(o_1),\qquad \delta_{t+1}(i) = b_i(o_{t+1}) \max_j \big[\delta_t(j)\, a_{ji}\big],\qquad
\psi_{t+1}(i) = \arg\max_j \big[\delta_t(j)\, a_{ji}\big].$$

*In words:* the best prefix ending in $s_i$ at $t+1$ is the best prefix ending in *some* $s_j$ at $t$, extended by the
arrow $j\to i$ and the emission of $o_{t+1}$; we pick the $j$ that makes this largest and **remember it** in the
**backpointer** $\psi_{t+1}(i)$. (The max can be taken inside because all factors are non-negative and the future
depends on the past only through the current state.)

**Step 3 — termination and traceback.** $P(Q^*, O) = \max_i \delta_T(i)$; the last state is $q_T^* = \arg\max_i \delta_T(i)$;
then follow the backpointers: $q_t^* = \psi_{t+1}(q_{t+1}^*)$ for $t = T-1, \dots, 1$.

⚠️ **Two common slips.**
* $\delta_t(i)$ is sometimes written as $\max_{q_1..q_t} P(q_1,\dots,q_t \mid o_1,\dots,o_t)$. The recursion above
  actually computes the **joint** $P(q_1..q_t, o_1..o_t)$, and the max runs over $q_1..q_{t-1}$ with $q_t = s_i$ held
  fixed (otherwise $\delta_t$ could not depend on $i$). Dividing by $P(o_1..o_t)$ would not change any argmax, which is
  why both versions find the same path.
* "$Q^*$ = the path defined by $\arg\max_j \delta_t(j)$" is a tempting shortcut. Taking the best state *separately at each
  $t$* is **not** the Viterbi path in general — you must trace the backpointers from the end. We show a two-roll
  counterexample below.

🧮 **Math refresher — why logarithms?** $\log(xy) = \log x + \log y$, and $\log$ is increasing, so the path that maximises a
product of probabilities also maximises the sum of their logs. Sums of logs never underflow (Section 10), so Viterbi is
always implemented in log-space: $\log\delta_{t+1}(i) = \log b_i(o_{t+1}) + \max_j[\log\delta_t(j) + \log a_{ji}]$.

### 9.1 A Viterbi table by hand, then an example sequence, honestly

After defining `viterbi`, we first fill its table by hand for the short sequence $O=(2,3,6)$ of Section 5 — $\delta$
values *and* backpointers $\psi$ — and draw it as a trellis. Then we turn to a classic exercise: what is the most
probable set of states behind the rolls `1 2 2 1 1 2 1 1 1 2 2`, with the dice parameters above? A tempting answer is
`A A A B B B B B B A A` — the same pattern as the hidden path of the story in Section 3.2, with a B-run in the middle.
We run Viterbi and then check by brute force over all $2^{11} = 2{,}048$ paths.

In [ ]:
# ── 9.1 The Viterbi algorithm (log-space) ──
def viterbi(obs, pi, trans, emit):
    """Most probable state path. Returns (path, log_delta, backpointers); log P(Q*, O) = log_delta[-1].max()."""
    T, n = len(obs), len(pi)
    with np.errstate(divide="ignore"):                       # log(0) = −inf is fine here
        log_pi, log_a, log_b = np.log(pi), np.log(trans), np.log(emit)
    log_delta = np.zeros((T, n))
    back = np.zeros((T, n), dtype=int)
    log_delta[0] = log_pi + log_b[:, obs[0]]                 # δ_1(i) = π_i b_i(o_1)
    for t in range(1, T):
        scores = log_delta[t - 1][:, None] + log_a           # scores[j, i] = log δ_t(j) + log a_ji
        back[t] = scores.argmax(axis=0)                      # ψ_{t+1}(i): best predecessor of each state i
        log_delta[t] = scores.max(axis=0) + log_b[:, obs[t]]
    path = np.zeros(T, dtype=int)
    path[-1] = log_delta[-1].argmax()                        # best final state
    for t in range(T - 1, 0, -1):                            # traceback along the backpointers
        path[t - 1] = back[t, path[t]]
    return path, log_delta, back

In [ ]:
# ── 9.1b The Viterbi table by hand for O = (2, 3, 6): δ values, backpointers ψ, traceback ──
a_ = {("A", "A"): 0.8, ("A", "B"): 0.2, ("B", "A"): 0.2, ("B", "B"): 0.8}     # a_ji = P(next = i | previous = j)
b_ = {"A": bA, "B": bB}                                                       # emission tables from cell 5.2
delta = [{"A": 0.7 * bA[2], "B": 0.3 * bB[2]}]
psi = [None]
print(f"t=1: δ1(A) = π_A·b_A(2) = 0.7 × 0.2 = {delta[0]['A']:.4g}    δ1(B) = π_B·b_B(2) = 0.3 × 0.1 = {delta[0]['B']:.4g}")
for t, o in enumerate([3, 6], start=2):
    new_d, new_p = {}, {}
    for i in "AB":
        cand = {j: delta[-1][j] * a_[(j, i)] for j in "AB"}                   # best way into i from each j
        new_p[i] = max(cand, key=cand.get)
        new_d[i] = b_[i][o] * cand[new_p[i]]
        print(f"t={t}: δ{t}({i}) = b_{i}({o}) · max(δ{t-1}(A)·a_A{i} = {cand['A']:.4g}, δ{t-1}(B)·a_B{i} = {cand['B']:.4g})"
              f" = {b_[i][o]} × {cand[new_p[i]]:.4g} = {new_d[i]:.4g}    ψ{t}({i}) = {new_p[i]}")
    delta.append(new_d); psi.append(new_p)
last = max(delta[-1], key=delta[-1].get)
path_hand = [last]
for t in (2, 1):                                                              # follow the backpointers ψ3, ψ2
    path_hand.insert(0, psi[t][path_hand[0]])
print(f"\ntermination: max_i δ3(i) = δ3({last}) = {delta[-1][last]:.4g};  traceback → Q* = {''.join(path_hand)}")
print(f"P(Q* | O) = {delta[-1][last]:.4g} / P(O) = {delta[-1][last] / alpha236[-1].sum():.3f}   (Section 7: AAA, 0.385)")
print("viterbi() returns:", "".join(STATES[s] for s in viterbi(O236, pi, trans, emit)[0]))


def draw_viterbi_trellis(ax, obs, log_delta, back, path, title):
    """Trellis of δ values with each node's backpointer (black arrow) and the traceback path (thick red)."""
    T, n = log_delta.shape
    draw_trellis(ax, np.exp(log_delta), obs, title)
    for t in range(1, T):
        for i in range(n):
            j = back[t, i]
            ax.annotate("", xy=(t - 0.27, n - 1 - i), xytext=(t - 1 + 0.27, n - 1 - j),
                        arrowprops=dict(arrowstyle="<-", color="0.15", lw=1.0, shrinkA=0, shrinkB=0))
    ax.plot(range(T), [n - 1 - s for s in path], color="tab:red", lw=6, alpha=0.35, zorder=1.5,
            solid_capstyle="round")


_, logd236, back236 = viterbi(O236, pi, trans, emit)
fig, ax = plt.subplots(figsize=(6.5, 2.8))
draw_viterbi_trellis(ax, O236, logd236, back236, [STATES.index(s) for s in path_hand],
                     "Viterbi table δ for O = (2, 3, 6): arrows = backpointers ψ, red = traceback")
plt.tight_layout(); plt.show()

🔍 **What just happened?** The Viterbi table is the forward table with "max" in place of "sum": $\delta_1=(.14,.03)$ as
before; $\delta_2(A)=.2\times\max(.8\times.14,\ .2\times.03)=.2\times.112=.0224$ and
$\delta_2(B)=.1\times\max(.2\times.14,\ .8\times.03)=.1\times.028=.0028$ — both best reached from A; $\delta_3(A)=.001792$
and $\delta_3(B)=.001344$, again both from A. The largest entry of the last column is $\delta_3(A)$, and following the
backpointers (red) gives $Q^*=$ AAA with $P(Q^*,O)=.001792$ and $P(Q^*\mid O)=.001792/.004656=0.385$ — exactly the best
of the 8 paths listed in Section 7. Each $\delta_t(i)$ is the joint probability of the best path *ending* in $i$: the
best path ending in B is AAB ($.001344$, the runner-up of Section 7).

Now the 11-roll example.

In [ ]:
# ── 9.2 Viterbi on the example sequence 1 2 2 1 1 2 1 1 1 2 2 ──
path_v, logd_v, back_v = viterbi(O_example, pi, trans, emit)
Q_tempting = to_states("AAABBBBBBAA")                 # the tempting answer
p_vit = np.exp(logd_v[-1].max())
p_tempt = joint_prob(Q_tempting, O_example, pi, trans, emit)
print("observations :", " ".join(str(o + 1) for o in O_example))
print("Viterbi path :", " ".join(STATES[s] for s in path_v))
print("tempting path:", " ".join(STATES[s] for s in Q_tempting))
print(f"\nP(Q*, O)          = {p_vit:.4g}")
print(f"P(tempting path, O) = {p_tempt:.4g}   → the Viterbi path is {p_vit / p_tempt:,.0f} times more probable")

In [ ]:
# ── 9.3 Brute-force check over all 2,048 paths ──
best = np.argmax(joint11)
idx_tempt = int("".join(map(str, Q_tempting)), 2)      # paths are listed in binary order: AAA…=0, …, BBB…=2^T−1
n_better = int(np.sum(joint11 > joint11[idx_tempt] * (1 + 1e-9)))   # tolerance: ignore rounding-level differences
P_O = joint11.sum()
print("brute-force best path:", "".join(STATES[s] for s in paths11[best]), f"with P(Q, O) = {joint11[best]:.4g}")
print(f"paths with a higher P(Q, O) than the tempting path: {n_better} of {len(joint11)}")
print(f"P(Q* | O) = {joint11[best] / P_O:.3f};   P(tempting path | O) = {p_tempt / P_O:.2e}")
gamma_example = posterior_states(O_example, pi, trans, emit)
print("posterior P(q_t = B | O):", np.round(gamma_example[:, 1], 3))

fig, ax = plt.subplots(figsize=(6.5, 3))
ax.hist(np.log10(joint11), bins=40, color="0.7")
ax.axvline(np.log10(joint11[best]), color="tab:blue", lw=2, label="Viterbi path (all A)")
ax.axvline(np.log10(p_tempt), color="tab:red", lw=2, ls="--", label="tempting path AAABBBBBBAA")
ax.set_xlabel(r"$\log_{10} P(Q, O)$"); ax.set_ylabel("number of paths")
ax.set_xlim(None, np.log10(joint11[best]) + 0.6)
ax.set_title("all 2,048 paths for O = 1 2 2 1 1 2 1 1 1 2 2"); ax.legend(loc="upper left")
plt.tight_layout(); plt.show()

🔍 **What just happened — honestly.** With the two-dice parameters, the most probable path
for `1 2 2 1 1 2 1 1 1 2 2` is **all A**, with $P(Q^*, O) = 1.75\times10^{-8}$; the tempting path
`A A A B B B B B B A A` has $P = 2.25\times10^{-12}$, is 7,776 times less probable, and 233 of the 2,048 paths are more probable than it. Brute
force confirms the Viterbi answer, and the posterior probability of die B stays below 0.18 at every position.

The ratio has a simple explanation you can do by hand. Relabelling positions 4–9 from A to B changes:
* the emissions of the faces there (1, 1, 2, 1, 1, 1): each "1" costs $b_B(1)/b_A(1) = 0.1/0.3 = 1/3$, the "2" costs
  $0.1/0.2 = 1/2$, so a factor $(1/3)^5 \times 1/2 = 1/486$;
* the transitions: two switches (0.2 each) replace two stays (0.8 each), a factor $(0.2/0.8)^2 = 1/16$.

Total: $1/(486 \times 16) = 1/7{,}776$. Faces 1 and 2 are *evidence for die A*, so nothing in these data supports a
B-segment. The tempting path has the same letters as the hidden path of the story in Section 3.2, i.e. it is *a* path
that could have generated rolls like these — not the argmax. (A second version of this exercise uses
$\pi_A=\pi_B=0.5$; the Viterbi path is still all A — try it in the ✏️ below.) What would make the decoder switch to B?
Rolls that die B explains better:

In [ ]:
# ── 9.4 A sequence whose middle really looks like die B ──
O_bseg = to_obs("1 2 2 6 5 6 4 6 6 2 2")
path_b, logd_b, _ = viterbi(O_bseg, pi, trans, emit)
gamma_b = posterior_states(O_bseg, pi, trans, emit)
print("observations :", " ".join(str(o + 1) for o in O_bseg))
print("Viterbi path :", " ".join(STATES[s] for s in path_b), f"  P(Q*, O) = {np.exp(logd_b[-1].max()):.3g}")
print("P(q_t=B | O) :", np.round(gamma_b[:, 1], 2))

fig, axes = plt.subplots(1, 2, figsize=(11, 2.9), sharey=True)
for ax, o, g, pth, ttl in [(axes[0], O_example, gamma_example, path_v, "example rolls 1 2 2 1 1 2 1 1 1 2 2"),
                           (axes[1], O_bseg, gamma_b, path_b, "rolls with a high-face middle")]:
    t_ax = np.arange(1, len(o) + 1)
    ax.bar(t_ax, g[:, 1], color="tab:orange", alpha=0.6)
    for t, f, st in zip(t_ax, o, pth):
        ax.text(t, 1.17, str(f + 1), ha="center", fontsize=9)                         # observed face
        ax.text(t, 1.03, STATES[st], ha="center", fontsize=10, weight="bold", color=DIE_COLORS[st])   # Viterbi state
    ax.set_xticks(t_ax); ax.set_xlabel("roll t")
    ax.set_title(ttl, pad=26)
axes[0].set_ylabel(r"$P(q_t = B \mid O)$"); axes[0].set_ylim(0, 1.28); axes[0].set_yticks([0, 0.25, 0.5, 0.75, 1])
fig.suptitle("top row: observed face · coloured letter: Viterbi state · bar: posterior probability of die B", fontsize=9)
plt.tight_layout(); plt.show()

🔍 **What just happened?** Replace the middle rolls by faces that die B favours (6, 5, 6, 4, 6, 6) and Viterbi returns
exactly the tempting pattern `A A A B B B B B B A A`, with posterior probabilities of B around 0.8–0.9 in the middle. The
algorithm is not biased against B — it follows the evidence. Notice also that the posteriors are *less* confident at the
edges of the B-run (0.36 at roll 3, 0.48 at roll 10): the exact switch points are genuinely uncertain.

### 9.2 Why the traceback matters

Here is the promised counterexample to "take $\arg\max_j\delta_t(j)$ at each $t$". Two rolls, both a 4.

In [ ]:
# ── 9.5 Per-position argmax of δ is not the Viterbi path ──
O44 = to_obs("4 4")
path44, logd44, back44 = viterbi(O44, pi, trans, emit)
print("δ table (rows t = 1, 2; columns A, B):\n", np.round(np.exp(logd44), 4))
print("argmax_j δ_t(j) separately at each t:", "".join(STATES[i] for i in logd44.argmax(axis=1)))
print("Viterbi path (traceback)            :", "".join(STATES[i] for i in path44))
for p in itertools.product(range(2), repeat=2):
    print(f"  P(Q = {''.join(STATES[s] for s in p)}, O = 4 4) = {joint_prob(np.array(p), O44, pi, trans, emit):.4f}")
fig, ax = plt.subplots(figsize=(4.6, 2.8))
draw_viterbi_trellis(ax, O44, logd44, back44, path44, "O = (4, 4): ψ2(B) = B corrects the early lead of A")
plt.tight_layout(); plt.show()

🔍 **What just happened?** After the first roll, $\delta_1(A) = 0.7\times0.1 = 0.07$ beats $\delta_1(B) = 0.3\times0.2 = 0.06$
(die A starts more often). But the best *two-roll* path is BB ($0.0096$): the second 4 is better explained by B, and
switching is expensive, so the best path to B at $t = 2$ comes *from B*. Picking the best state at each time separately
gives **AB**, whose joint probability (0.0028) is less than a third of BB's. The backpointer $\psi_2(B) = B$ is what
corrects the early decision. **Always trace back.**

### 9.3 Viterbi vs posterior decoding on a long sequence

In [ ]:
# ── 9.6 Compare the two decoders on the 200 simulated rolls of Section 8 ──
q_vit, logd_long, _ = viterbi(o_long, pi, trans, emit)
print(f"accuracy: Viterbi {accuracy(q_true, q_vit):.3f} | posterior decoding {accuracy(q_true, q_post):.3f}")
print(f"positions where the two decoders disagree: {np.sum(q_vit != q_post)} of {T_long}")
n_runs = lambda q: 1 + int(np.sum(q[1:] != q[:-1]))
print(f"number of runs: truth {n_runs(q_true)}, Viterbi {n_runs(q_vit)}, posterior decoding {n_runs(q_post)}")
print(f"log P(Q*, O) = {logd_long[-1].max():.2f}   vs   log P(posterior-decoded path, O) = "
      f"{np.log(joint_prob(q_post, o_long, pi, trans, emit)):.2f}")

🔍 **What just happened?** Posterior decoding is slightly more accurate (0.735 vs 0.715) and the two decoders disagree
at 32 of the 200 positions. Viterbi returns only 13 runs where the truth has 38: short runs are not worth the two
"switch" penalties ($0.2$ each instead of $0.8$), so the most probable *path* merges them away. They optimise different things: **posterior decoding** maximises the *expected number
of correctly labelled positions*; **Viterbi** maximises the probability of the *whole path* being right. Viterbi's output
is always a coherent path, while posterior decoding can produce rapid flips (43 runs here) — and in
models with forbidden transitions ($a_{ij}=0$, e.g. "an intron cannot follow intergenic background directly") it can
even produce an *impossible* path. And the joint probability of the posterior-decoded path can never exceed Viterbi's
(log values −417.6 vs −400.0 in the last line).
Gene finders therefore report the Viterbi parse, often together with posterior probabilities as confidence scores.

✏️ **Your turn.** A second version of the example uses the same model with $\pi_A = \pi_B = 0.5$. Run
`viterbi(O_example, np.array([0.5, 0.5]), trans, emit)`. Does the most probable path change? Then find (by trial) the
smallest number of 6s you must put in the middle of `O_example` before Viterbi starts a B-segment.

### 📘 Lecture notes — posterior decoding and the Viterbi algorithm

**Posterior of one position** (definition of conditional probability; product rule; Markov property; law of total
probability):

$$P(q_t=A\mid o_1,\dots,o_T)=\frac{P(o_1..o_t,q_t=A)\,P(o_{t+1}..o_T\mid q_t=A)}{P(o_1..o_T)}=\frac{\alpha_t(A)\,\beta_t(A)}{\sum_i\alpha_t(i)\,\beta_t(i)}.$$

*In words:* "past ending in A" × "future starting from A", normalised. Example $O=(2,3,6)$:
$P(q_t=A\mid O)=.830,\ .710,\ .428$ → posterior decoding A A B.

**Most probable path** $Q^*=\arg\max_QP(Q\mid O)$ — the words in speech, the set of genes in a genome. Since $P(O)$
does not depend on $Q$: $\arg\max_QP(Q\mid O)=\arg\max_QP(O\mid Q)P(Q)=\arg\max_QP(O,Q)$.

**Viterbi algorithm.** $\delta_t(i)$ = maximum probability of ending in state $s_i$ at time $t$ (over all paths up to
$t$): $\delta_1(i)=P(o_1\mid q_1=s_i)P(q_1=s_i)$;

$$\delta_{t+1}(i)=P(o_{t+1}\mid q_{t+1}=s_i)\,\max_j\big[P(q_{t+1}=s_i\mid q_t=s_j)\,\delta_t(j)\big],\qquad
\psi_{t+1}(i)=\arg\max_j[\dots].$$

*In words:* the forward table with $\sum_j$ replaced by $\max_j$, plus a backpointer $\psi$ that remembers the best
predecessor. Finish with $P(Q^*,O)=\max_i\delta_T(i)$ and trace the backpointers back from $\arg\max_i\delta_T(i)$.
Example $O=(2,3,6)$: $\delta_1=(.14,.03)$, $\delta_2=(.0224,.0028)$, $\delta_3=(.001792,.001344)$, all backpointers A →
$Q^*=$ AAA, $P(Q^*\mid O)=.385$ — while posterior decoding gives A A B.

**The example `1 2 2 1 1 2 1 1 1 2 2`.** Its most probable path is all A ($P(Q^*,O)=1.75\times10^{-8}$); the tempting
`A A A B B B B B B A A` is $7{,}776$ times less probable: $(1/3)^5\cdot\frac12$ for the emissions of rolls 4–9 times
$(0.2/0.8)^2$ for two switches.

**Common confusions.** Taking $\arg\max_j\delta_t(j)$ separately at each $t$ is not the Viterbi path (counterexample
$O=(4,4)$: per-position AB, Viterbi BB). Posterior decoding maximises the expected number of correct positions,
Viterbi the probability of the whole path.

> 🎓 **Exam tip.** Viterbi = forward algorithm with $\sum$ replaced by $\max$ **plus backpointers and traceback**. In an exam, show the $\delta$ table *and* the $\psi$ table, and remember that the most probable path need not agree with the position-wise most probable states.

---
## 10 · Numerical underflow, and two ways out

> 📍 **Where we are.** Everything so far used sequences of at most a few hundred symbols. Real sequences (a gene, a
> chromosome, a long time series) have thousands to millions. The algorithms are fine; the *numbers* are not.

Each forward step multiplies by probabilities below 1, so $P(O)$ shrinks roughly geometrically with $T$ — for the dice
by a factor of about 6 per roll (one face out of six). A 64-bit floating-point number cannot represent positive values
below about $10^{-308}$ (or $5\times10^{-324}$ with reduced precision); anything smaller becomes exactly 0. This is
**underflow**.

In [ ]:
# ── 10.1 Watch the naive forward algorithm underflow ──
rng = np.random.default_rng(SEED + 10)
_, o_huge = sample_hmm(5000, pi, trans, emit, rng)
for T in [100, 200, 300, 400, 450, 500, 1000]:
    print(f"T = {T:5d}: naive P(O) = {forward(o_huge[:T], pi, trans, emit)[-1].sum():.3e}")

🔍 **What just happened?** $P(O)$ drops by roughly 77 orders of magnitude per 100 rolls (from $\sim10^{-78}$ at
$T = 100$ to $\sim10^{-233}$ at $T = 300$; the exact values are printed); somewhere between $T = 400$ and $450$ it falls
off the bottom of the floating-point range and becomes exactly 0. From then on every posterior is $0/0$. Two standard
cures:

**Cure 1 — scaling (Rabiner 1989).** Normalise each forward column to sum to 1 and remember the normaliser:

$$c_t = \sum_i \Big[b_i(o_t)\sum_j \hat\alpha_{t-1}(j)\,a_{ji}\Big], \qquad \hat\alpha_t(i) = \frac{b_i(o_t)\sum_j \hat\alpha_{t-1}(j)\,a_{ji}}{c_t},
\qquad \log P(O) = \sum_{t=1}^T \log c_t .$$

*In words:* $c_t = P(o_t \mid o_1,\dots,o_{t-1})$ is the probability of the *next* observation given the previous ones
(a number of reasonable size), $\hat\alpha_t(i) = P(q_t = s_i \mid o_1..o_t)$ is a proper probability distribution over
states, and the product rule $P(O) = \prod_t P(o_t\mid o_1..o_{t-1})$ turns into a sum of logs.

**Cure 2 — log-space.** Store $\log\alpha_t(i)$ and replace "add probabilities" by "log-sum-exp":
$\log(e^x + e^y) = m + \log(e^{x-m} + e^{y-m})$ with $m = \max(x,y)$ — the toolkit's `logsumexp`. Viterbi only needs max
and +, so its log version (Section 9) never underflows.

In [ ]:
# ── 10.2 Scaled and log-space forward algorithms ──
def forward_scaled(obs, pi, trans, emit):
    """Scaled forward pass: alpha_hat[t] = P(q_t | o_1..o_t) (rows sum to 1) and c[t] = P(o_t | o_1..o_{t-1})."""
    T, n = len(obs), len(pi)
    alpha_hat, c = np.zeros((T, n)), np.zeros(T)
    a = pi * emit[:, obs[0]]
    c[0] = a.sum(); alpha_hat[0] = a / c[0]
    for t in range(1, T):
        a = emit[:, obs[t]] * (alpha_hat[t - 1] @ trans)
        c[t] = a.sum(); alpha_hat[t] = a / c[t]
    return alpha_hat, c                                      # log P(O) = np.log(c).sum()


def backward_scaled(obs, trans, emit, c):
    """Backward pass scaled with the same c[t]; then alpha_hat * beta_hat = P(q_t | O) directly."""
    T, n = len(obs), trans.shape[0]
    beta_hat = np.ones((T, n))
    for t in range(T - 2, -1, -1):
        beta_hat[t] = trans @ (emit[:, obs[t + 1]] * beta_hat[t + 1]) / c[t + 1]
    return beta_hat


def forward_log(obs, pi, trans, emit) -> float:
    """log P(O) with the forward recursion carried out on logarithms (log-sum-exp instead of sums)."""
    log_a, log_b = np.log(trans), np.log(emit)
    log_alpha = np.log(pi) + log_b[:, obs[0]]
    for t in range(1, len(obs)):
        log_alpha = log_b[:, obs[t]] + logsumexp(log_alpha[:, None] + log_a, axis=0)
    return logsumexp(log_alpha)


for T in [200, 1000, 5000]:
    o = o_huge[:T]
    with np.errstate(divide="ignore"):
        naive = np.log(forward(o, pi, trans, emit)[-1].sum())
    _, c = forward_scaled(o, pi, trans, emit)
    print(f"T = {T:4d}: log P(O) naive {naive:10.3f} | scaled {np.log(c).sum():10.3f} | log-space {forward_log(o, pi, trans, emit):10.3f}")

ah, c = forward_scaled(o_long, pi, trans, emit)
gamma_scaled = ah * backward_scaled(o_long, trans, emit, c)
print(f"\nscaled posteriors vs unscaled (200 rolls of Section 8): max difference {np.abs(gamma_scaled - gamma_long).max():.1e}")

In [ ]:
# ── 10.3 Picture: where the naive algorithm breaks ──
Ts = np.arange(25, 1001, 25)
with np.errstate(divide="ignore"):
    naive_l10 = [np.log10(forward(o_huge[:T], pi, trans, emit)[-1].sum()) for T in Ts]
scaled_l10 = [np.log(forward_scaled(o_huge[:T], pi, trans, emit)[1]).sum() / np.log(10) for T in Ts]
fig, ax = plt.subplots(figsize=(6.5, 3.2))
ax.plot(Ts, scaled_l10, color="tab:blue", lw=2, label="scaled (correct)")
ax.plot(Ts, naive_l10, "o", color="tab:red", ms=3.5, label="naive (−∞ once P(O) underflows to 0)")
ax.axhline(-308, color="0.5", ls="--", lw=1); ax.text(560, -290, "smallest normal float64 ≈ 1e-308", fontsize=8, color="0.3")
ax.set_xlabel("sequence length T"); ax.set_ylabel(r"$\log_{10} P(O)$")
ax.set_title("SIMULATED dice rolls: the naive forward algorithm underflows"); ax.legend(loc="lower left")
plt.tight_layout(); plt.show()

🔍 **What just happened?** All three methods agree at $T = 200$. At $T = 1{,}000$ and $5{,}000$ the naive version returns
$-\infty$ (it computed $P(O) = 0$), while the scaled and log-space versions agree to the printed digits. The scaled
posteriors equal the unscaled ones where both work. From now on — and in Module 20 — we use the scaled version, which
also gives the posteriors directly as $\hat\alpha_t(i)\,\hat\beta_t(i)$.

✏️ **Your turn.** Time `forward_scaled` and `forward_log` on the 5,000-roll sequence with `time.perf_counter()`. Which is
faster, and why might `logsumexp` be slower than a normalisation?

---
## 11 · Biology: finding CpG-island-like regions in a SIMULATED genome

> 📍 **Where we are.** We have all three inference algorithms, in a numerically safe form. Now the same code, unchanged,
> on a genomics problem — with four letters instead of six faces.

🧫 **Background.** In vertebrate genomes the dinucleotide CG ("CpG") is rare: methylated cytosines in CpG tend to mutate
to thymine (spontaneous deamination), so CpGs are slowly lost. **CpG islands** are stretches of a few hundred to a few
thousand bases where CpGs are *not* depleted and G + C content is high; they often mark promoters of genes. A classical
operational definition (Gardiner-Garden & Frommer 1987): length ≥ 200 bp, G + C > 50 %, observed/expected CpG ratio > 0.6.
Finding islands is the textbook HMM example of Durbin *et al.* (1998, Chapter 3).

**Our toy (SIMULATED, simplified).** Two hidden states, *background* and *island*. Real islands are defined by
*dinucleotide* (CG) frequencies, which Durbin *et al.* model with an 8-state HMM; to keep the code identical to the dice
we let the states differ only in single-base composition: background A .30, C .20, G .20, T .30; island A .15, C .35,
G .35, T .15. A background base enters an island with probability 1/3000 and an island base leaves with probability
1/500, so islands last ~500 bp on average and are separated by ~3 kb. These numbers are illustrative, not estimates.

In [ ]:
# ── 11.1 Simulate a 30-kb toy genome ──
BASES = "ACGT"
pi_cpg = np.array([0.95, 0.05])                                 # 0 = background, 1 = island
trans_cpg = np.array([[1 - 1 / 3000, 1 / 3000],
                      [1 / 500, 1 - 1 / 500]])
emit_cpg = np.array([[0.30, 0.20, 0.20, 0.30],                  # background: A, C, G, T
                     [0.15, 0.35, 0.35, 0.15]])                 # island: G+C rich

rng = np.random.default_rng(SEED + 11)
q_dna, o_dna = sample_hmm(30_000, pi_cpg, trans_cpg, emit_cpg, rng)
starts = np.flatnonzero(np.diff(np.r_[0, q_dna]) == 1)
ends = np.flatnonzero(np.diff(np.r_[q_dna, 0]) == -1) + 1
print("SIMULATED genome:", "".join(BASES[b] for b in o_dna[:70]), "…")
print(f"{len(o_dna):,} bp; {len(starts)} islands, lengths {(ends - starts).tolist()} bp; "
      f"{q_dna.mean():.1%} of bases in islands")
print(f"G+C content: background {np.isin(o_dna[q_dna == 0], [1, 2]).mean():.3f}, islands {np.isin(o_dna[q_dna == 1], [1, 2]).mean():.3f}")

In [ ]:
# ── 11.2 Decode with the HMM and compare with two baselines ──
ah, c = forward_scaled(o_dna, pi_cpg, trans_cpg, emit_cpg)
post_island = (ah * backward_scaled(o_dna, trans_cpg, emit_cpg, c))[:, 1]
pred_post = (post_island > 0.5).astype(int)
pred_vit, _, _ = viterbi(o_dna, pi_cpg, trans_cpg, emit_cpg)

# baseline 1: classify each base on its own (no Markov chain): island iff b_island(o) > b_background(o)
pred_base = (emit_cpg[1, o_dna] > emit_cpg[0, o_dna]).astype(int)
# baseline 2: G+C fraction in a sliding 200-bp window > 0.5 (like the island definition)
gc = np.convolve(np.isin(o_dna, [1, 2]).astype(float), np.ones(200) / 200, mode="same")
pred_win = (gc > 0.5).astype(int)

rows = []
for name, pred in [("each base alone", pred_base), ("200-bp window G+C > 50 %", pred_win),
                   ("HMM posterior > 0.5", pred_post), ("HMM Viterbi", pred_vit)]:
    p, r, f1 = precision_recall_f1(q_dna, pred)
    rows.append((name, accuracy(q_dna, pred), p, r, f1))
print(pd.DataFrame(rows, columns=["method", "accuracy", "precision", "recall", "F1"]).round(3).to_string(index=False))
print(f"\nlog P(genome | model) = {np.log(c).sum():,.1f} nats")

In [ ]:
# ── 11.3 Look at a 12-kb stretch ──
lo, hi = 1000, 13000
x = np.arange(lo, hi)
fig, axes = plt.subplots(3, 1, figsize=(10, 4.6), sharex=True)
axes[0].plot(x, gc[lo:hi], color="0.4", lw=0.8); axes[0].axhline(0.5, color="tab:red", ls="--", lw=0.8)
axes[0].set_ylabel("G+C\n(200-bp window)")
axes[1].fill_between(x, 0, q_dna[lo:hi], step="mid", color="tab:green", alpha=0.3, label="true island (hidden)")
axes[1].plot(x, post_island[lo:hi], color="tab:blue", lw=1, label="P(island | sequence)")
axes[1].set_ylabel("posterior"); axes[1].legend(loc="upper left", fontsize=7, ncol=2)
axes[2].fill_between(x, 0, q_dna[lo:hi], step="mid", color="tab:green", alpha=0.3, label="truth")
axes[2].step(x, pred_vit[lo:hi] * 0.8, where="mid", color="tab:purple", lw=1.2, label="Viterbi (drawn at 0.8)")
axes[2].set_ylabel("segments"); axes[2].set_xlabel("position in SIMULATED genome (bp)")
axes[2].legend(loc="upper left", fontsize=7, ncol=2)
for ax in axes[1:]:
    ax.set_ylim(-0.05, 1.05)
axes[0].set_title("SIMULATED CpG-island-like regions: the HMM sharpens the noisy G+C signal")
plt.tight_layout(); plt.show()

🔍 **What just happened?** Classifying each base on its own is useless: about 61 % accuracy with many false islands —
far worse than simply calling every base background, which would be right about 89 % of the time (only 11 % of the bases
are in islands). A single C or G is weak evidence. A sliding-window rule — essentially the island definition — is
already good. The HMM does better still on every measure (the table above gives the exact values; on this simulated
genome both HMM decoders exceed 99 % accuracy), and its segment boundaries are sharp: the transition probabilities
tell it that islands are long and rare, so it neither flickers in and out nor needs an arbitrary window size. On real
genomes the comparison is less flattering: real island composition varies, real boundaries are fuzzy, and the
transition probabilities must be *learned* — the subject of Module 20. An HMM is only as good as its parameters.

⚠️ **The model is an assumption, not a discovery.** We decoded data generated *by the same model*. In real data the
geometric distribution of island lengths implied by a constant leaving probability (Module 20, semi-Markov models), the
single-base emissions and the two-state picture are all approximations; posterior probabilities are only as calibrated
as the model is correct.

✏️ **Your turn.** (1) Make the islands weaker — island emissions `[0.2, 0.3, 0.3, 0.2]` — and re-run 11.1–11.2: which
method suffers most? (2) Decode with a *wrong* leaving
probability, 1/50 instead of 1/500, but the correct emissions. What happens to the number of predicted islands?

---
## 📌 Key takeaways

What you should know about HMMs so far — why HMMs, and inference (the learning part follows in Module 20):

- **Why HMMs?** Sequences whose hidden labels persist and "speak" in characteristic ways — genes, CpG islands, speech,
  haplotypes, dice. An HMM = a Markov chain of hidden states + an emission table, i.e. a chain-shaped Bayesian network
  with $P(Q,O) = \pi_{q_1} b_{q_1}(o_1) \prod_{t\ge2} a_{q_{t-1}q_t} b_{q_t}(o_t)$ and only $(n-1)+n(n-1)+n(m-1)$ parameters.
- **Conditional independence buys tractability**: observations are dependent, but independent given the hidden chain;
  that is what lets the $n^T$-term sum $P(O) = \sum_Q P(Q,O)$ be computed in $O(n^2T)$.
- **Forward**: $\alpha_t(i) = P(o_1..o_t, q_t = s_i)$, $\alpha_{t+1}(i) = b_i(o_{t+1})\sum_j \alpha_t(j)a_{ji}$,
  $P(O) = \sum_i\alpha_T(i)$. **Backward**: $\beta_t(i) = P(o_{t+1}..o_T\mid q_t = s_i)$,
  $\beta_t(i) = \sum_j a_{ij} b_j(o_{t+1})\beta_{t+1}(j)$, $\beta_T = 1$. Both derived with three rules only: law of total
  probability, product rule, Markov property.
- **Various inference tasks**: $P(Q\mid O)$ for one path (easy numerator, forward denominator); posterior decoding
  $P(q_t = s_i\mid O) = \alpha_t(i)\beta_t(i)/P(O)$; $\sum_i\alpha_t(i)\beta_t(i) = P(O)$ at every $t$ is the unit test.
- **Viterbi**: forward with max instead of sum, plus backpointers and traceback. It maximises the probability of the
  whole path; posterior decoding maximises the expected number of correct positions — they can disagree.
- **Honesty about the example sequence**: with the two-dice parameters the most probable path of
  `1 2 2 1 1 2 1 1 1 2 2` is all A ($1.75\times10^{-8}$ vs $2.25\times10^{-12}$ for the tempting B-run path
  `A A A B B B B B B A A`) — a plausible generating path is not necessarily the argmax.
- **Long sequences underflow**: use scaling ($\log P(O) = \sum_t\log c_t$) or log-space arithmetic.

---
## ✅ Checkpoint

1. Compute $\alpha_1$, $\alpha_2$ and $P(O)$ by hand for $O = (6, 6)$ with the two-dice parameters.
   <details><summary>Answer</summary>$\alpha_1(A) = .7\times.1 = .07$, $\alpha_1(B) = .3\times.3 = .09$; $\alpha_2(A) = .1\times(.8\times.07 + .2\times.09) = .0074$, $\alpha_2(B) = .3\times(.2\times.07 + .8\times.09) = .0258$; $P(O) = .0332$, slightly *more* than $P(2,3) = .0236 + .0052 = .0288$: a 6 is die B's favourite face (0.3) and die B tends to stay, so 6, 6 is a coherent 'die B' story, whereas 2 and 3 have probability only 0.2 each even under die A.</details>
2. In the forward derivation, which step uses the Markov property, and which conditional independence exactly?
   <details><summary>Answer</summary>Step 4: $P(o_{t+1}, q_{t+1} \mid o_1..o_t, q_t) = P(o_{t+1}, q_{t+1}\mid q_t)$ — given $q_t$, the next state and the next output are independent of the earlier observations. Step 6: $P(o_{t+1}\mid q_{t+1}, q_t) = P(o_{t+1}\mid q_{t+1})$ — an output depends only on its own state.</details>
3. Why is $\beta_T(i) = 1$, and what goes wrong if you initialise $\beta_T(i) = b_i(o_T)$?
   <details><summary>Answer</summary>$\beta_T(i)$ is the probability of the (empty) set of observations after $T$, which is 1. With $b_i(o_T)$ the last emission would be counted twice (it is already inside $\alpha_T$), so $\sum_i\alpha_t(i)\beta_t(i)$ would no longer equal $P(O)$.</details>
4. For $O = (4, 4)$, what is the Viterbi path, and what does 'argmax of $\delta_t$ at each $t$' give?
   <details><summary>Answer</summary>$\delta_1 = (.07, .06)$ favours A, but $P(BB, O) = .3\times.2\times.8\times.2 = .0096$ is the largest joint probability (AA: .0056, AB: .0028, BA: .0012). Viterbi (with traceback) returns BB; per-position argmax returns AB.</details>
5. Posterior decoding of $O = (2, 3, 6)$ gives AAB while the most probable path is AAA. Explain without formulas.
   <details><summary>Answer</summary>Posterior decoding chooses, for each position separately, the state with the largest total probability over all paths. At position 3 the paths ending in B (AAB, ABB, BBB, BAB) together have probability .572 > .5, although the single best path AAA (.385) ends in A.</details>
6. A gene finder has $n = 3$ states and a chromosome has $T = 10^8$ bases. Compare the cost of brute force and of the forward algorithm, and say what else you must do to run the forward algorithm.
   <details><summary>Answer</summary>Brute force: $3^{10^8}$ paths — impossible. Forward: $\approx n^2T = 9\times10^8$ multiply–adds, seconds to minutes. You must also scale (or work in logs), since $P(O)$ is far below $10^{-308}$.</details>
7. Code reading: in `forward`, what is the $i$-th entry of `alpha[t - 1] @ trans`, and why is it multiplied by `emit[:, obs[t]]`?
   <details><summary>Answer</summary>$\sum_j \alpha_t(j)\,a_{ji} = P(o_1..o_t, q_{t+1} = s_i)$ — the probability of the past observations and of having moved into $s_i$. Multiplying by $b_i(o_{t+1})$ adds the emission of the new observation, giving $\alpha_{t+1}(i)$.</details>
8. With the two-dice parameters, why is the Viterbi path of `1 2 2 1 1 2 1 1 1 2 2` all A, although the story-like path A A A B B B B B B A A looks natural?
   <details><summary>Answer</summary>Faces 1 and 2 are 3× and 2× more likely under die A, and switching costs 0.2 instead of 0.8. Relabelling rolls 4–9 as B multiplies the joint probability by $(1/3)^5(1/2)(0.2/0.8)^2 = 1/7776$. That path has the pattern of the generating path of the story in Section 3.2 — a plausible path rather than the argmax.</details>

## 🧠 Questions & discussion

1. The HMM assumes that the hidden state at $t+1$ depends only on the state at $t$. For gene finding, give a biological feature that violates this assumption (think of reading frames or exon lengths). How could the state space be enlarged to restore the Markov property?
2. Posterior probabilities of 0.5 appear around the switch points in Section 8. Is that a failure of the method or a feature? How would you report uncertainty in a genome annotation?
3. Viterbi returns one path; posterior decoding returns per-position labels. For a clinician interpreting a copy-number profile from a tumour (states: loss / normal / gain), which output would you want and why?
4. In Section 11 we decoded data simulated from the very model we used for decoding. List three ways real genomic data could differ from this model, and predict how each would affect the posterior probabilities.
5. The tempting path A A A B B B B B B A A of Section 9 is not the argmax. When teaching (or reading a paper), how can you tell whether a figure shows the *true* hidden states, a *sampled* path, or the *decoded* path? Why does it matter?

## 🚀 Bonus — going further

- **Sample paths from the posterior.** Implement forward-filtering backward-sampling: run the forward pass, draw $q_T \propto \alpha_T$, then $q_t \propto \alpha_t(i)\,a_{i,q_{t+1}}$ backwards. Draw 1,000 paths for the example sequence `1 2 2 1 1 2 1 1 1 2 2` and estimate how often the tempting path appears (compare with its $P(Q\mid O)$ printed in cell 9.3). This is the bridge to Gibbs sampling in [Module 21](21_bayesian_learning_mcmc.ipynb).
- **A three-state gene-finder toy.** Add an intron state with its own composition and forbid background → intron transitions ($a = 0$). Find a sequence for which posterior decoding produces a forbidden (impossible) path but Viterbi does not.
- **Durbin's 8-state CpG model.** Build states A+, C+, G+, T+ (island) and A−, C−, G−, T− (background), where each state emits its own letter with probability 1, so that transitions capture dinucleotide frequencies such as the CG depletion. Decode the toy genome and compare.
- **Log-space everything.** Write `backward_log` and a posterior computed entirely in logs; compare speed and accuracy with the scaled version on 100,000 simulated bases.
- For the statistical background (likelihood, MLE vs MAP) used to *learn* HMMs in Module 20, see the repository notebook [Fundamentals of learning under uncertainty](../../FundamentalsLearningUncertainty/); the sister course [BiologyToResearch](../../../../cryoem/tutorial/BiologyToResearch/) uses the same latent-variable/EM logic for cryo-ET.
- **Markov chains without hidden states, and beyond.** [Module 32](32_cell_dynamics_velocity.ipynb) uses Markov chains (transition matrices, stationary distributions) to model how cells move between states in differentiation; [Module 43](43_llms_ai_agents.ipynb) starts from the same idea — predicting the next symbol from the previous ones — and follows it to modern language models, which condition on long contexts instead of a single hidden state.

## 📝 Lab note

Copy this template into your lab notebook and fill it in.

```text
Module: 19 · Hidden Markov models I — inference
Input: an observation sequence O = (o_1..o_T) and HMM parameters (π, a, b) — dice rolls / DNA bases (SIMULATED)
Output: P(O); α and β tables; P(q_t = s_i | O) for every t; the Viterbi path Q*
Method: forward and backward recursions (dynamic programming); Viterbi = max-product + traceback; scaling/log-space
Key equation: α_{t+1}(i) = b_i(o_{t+1}) Σ_j α_t(j) a_ji ;  P(q_t = s_i | O) = α_t(i) β_t(i) / P(O)
Key parameters: number of states n, transition matrix (persistence), emission tables (how distinct the states are)
What changed when I modified the stickiness / the emission table / π: …
One failure mode: underflow of the naive forward algorithm for T ≳ 450 dice rolls; per-position argmax ≠ Viterbi path
Connection to the 02-620 topics (which idea of the lecture this illustrates): the HMM as a Bayesian network;
                             forward/backward recursions (O = {2,3,6}); posterior decoding; Viterbi (the most
                             probable path is not the tempting B-run path)
```

## 📚 References

- Carnegie Mellon University 02-620 *Machine Learning for Scientists* (course design S. Kim; instructors M. Xu and M. Zhang) — this module teaches the first part of its lecture on hidden Markov models and dynamic Bayesian networks (the model and inference).
- Rabiner L. R. (1989) A tutorial on hidden Markov models and selected applications in speech recognition. *Proceedings of the IEEE* 77:257–286.
- Durbin R., Eddy S. R., Krogh A., Mitchison G. (1998) *Biological Sequence Analysis: Probabilistic Models of Proteins and Nucleic Acids*. Cambridge University Press — Chapter 3 (Markov chains and HMMs; CpG islands; the dishonest casino).
- Murphy K. P. (2012) *Machine Learning: A Probabilistic Perspective*. MIT Press — Chapter 17 (Markov and hidden Markov models).
- Viterbi A. J. (1967) Error bounds for convolutional codes and an asymptotically optimum decoding algorithm. *IEEE Transactions on Information Theory* 13:260–269.
- Lander E. S. *et al.* (International Human Genome Sequencing Consortium) (2001) Initial sequencing and analysis of the human genome. *Nature* 409:860–921.
- Burge C., Karlin S. (1997) Prediction of complete gene structures in human genomic DNA. *Journal of Molecular Biology* 268:78–94 (GENSCAN).
- Krogh A., Mian I. S., Haussler D. (1994) A hidden Markov model that finds genes in *E. coli* DNA. *Nucleic Acids Research* 22:4768–4778.
- Gardiner-Garden M., Frommer M. (1987) CpG islands in vertebrate genomes. *Journal of Molecular Biology* 196:261–282.
- Ernst J., Kellis M. (2012) ChromHMM: automating chromatin-state discovery and characterization. *Nature Methods* 9:215–216.

## ➡️ Next

Here the parameters were given. In [Module 20 · Hidden Markov models II: learning, Baum–Welch and beyond](20_hmm_learning.ipynb) we learn them from data — by counting when the states are known, and with the EM (Baum–Welch) algorithm, which uses this notebook's forward–backward pass as its E-step, when they are hidden.